# Infinite OCR — full pipeline (router → route → one DOCX)

```
PDF
├─ Digital (text layer)
│   ├─ English      → Docling Heron layout → ordering prompt (Gemini, capped thinking) → tags → DOCX
│   └─ Non-English  → single Gemini call per page (OCR + order + tags) → [review later] → DOCX
└─ Non-digital / scanned
    ├─ English / Hindi → single Gemini call per page → [review later] → DOCX
    └─ Indic (Telugu, Tamil, …) → Bodhan OCR → dedup → LLM ordering → Qwen formatting tags → DOCX
```

**Run all cells top to bottom.** The router cell decides the route (override with `FORCE_ROUTE`); only that route's
cells do work, the others print `skip`. Every route ends in the **same tag grammar** (`<P> <STYLE> <LIST> <OPTIONS> <TABLE> <BLANK> <BR/>`)
and the **same DOCX builder**, and writes a sidecar `.blocks.json` (block id → page, bbox, rendered?) for the reviewer later.

Pick models in the **Config** cell (`MODEL_OPTIONS` + `CHOICE`). Reviewer is intentionally not included yet.


In [ ]:
# ─── CELL 1: setup & config ───────────────────────────────────────────────
# First run only:  !pip -q install pymupdf opencv-python-headless python-docx requests pillow docling
import base64, io, json, os, re, sys, time, string, datetime, hashlib, socket, math, difflib, statistics
import html as _html
from pathlib import Path
from collections import defaultdict, Counter
from concurrent.futures import ThreadPoolExecutor
import cv2
import numpy as np
import pymupdf
import requests
from PIL import Image, ImageDraw, ImageFont

from docx import Document
from docx.shared import Pt, Inches, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH, WD_COLOR_INDEX, WD_LINE_SPACING
from docx.enum.table import WD_TABLE_ALIGNMENT
from html import unescape

import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

# ── INPUT ──
PDF_PATH     = Path("pdf/tmax.pdf")   # change this
PAGES        = -1            # -1 = all, or 0-indexed list e.g. [0, 2, 5]
DPI          = 200
CHUNK_BLOCKS = 4             # Indic route: blocks per formatting call
FORCE_ROUTE  = None          # None = auto | "digital_english" | "single_gemini" | "scanned_indic"

# ── MODELS: one menu per step, pick with CHOICE (index into the list). Edit the lists freely. ──
MODEL_OPTIONS = {
    "classify":      ["qwen/qwen3-vl-30b-a3b-instruct","google/gemini-2.5-flash-lite", "google/gemini-2.5-flash", "qwen/qwen3-vl-30b-a3b-instruct"],            # script probe (scanned)
    "digital_order": ["google/gemini-2.5-flash", "google/gemini-2.5-flash-lite", "qwen/qwen3-vl-30b-a3b-instruct", "nvidia/nemotron-3.5-super-vl"],
    "single_call":   ["google/gemini-2.5-flash", "google/gemini-2.5-pro", "qwen/qwen3-vl-235b-a22b-instruct"],               # OCR+order+tags in one call
    "indic_order":   ["google/gemini-2.5-flash", "nvidia/nemotron-3.5-super-vl", "qwen/qwen3-vl-30b-a3b-instruct"],           # after Bodhan
    "indic_format":  ["google/gemini-2.5-flash","qwen/qwen3-vl-30b-a3b-instruct", "google/gemini-2.5-flash", "qwen/qwen3.5-9b"],                        # tags for DOCX ("Docx Qwen")
    "ocr_fallback":  ["google/gemini-2.5-flash", "qwen/qwen3-vl-30b-a3b-instruct"],                                           # if Bodhan fails
}
CHOICE = {"classify": 0, "digital_order": 0, "single_call": 0, "indic_order": 0, "indic_format": 0, "ocr_fallback": 0}
M = {k: MODEL_OPTIONS[k][CHOICE[k]] for k in MODEL_OPTIONS}

GEMINI_THINK_TOKENS = 512     # reasoning cap applied to every google/gemini* call (None = model default). Lower = faster.
# ── parallelism (threads; lower a knob if OpenRouter/Bodhan return many HTTP 429) ──
ORDER_WORKERS       = 8       # pages in parallel: digital ordering, single-Gemini calls, Indic ordering
BODHAN_WORKERS      = 4       # pages in parallel for Bodhan OCR (+dedup/label)
CHUNK_WORKERS       = 8       # Indic formatting chunks in parallel

# ── legacy names the Indic route (Stage 1/2 code) expects ──
OCR_FALLBACK_MODEL = M["ocr_fallback"]
COMBINED_MODELS    = [M["indic_format"]] + [m for m in ("google/gemini-2.5-flash",) if m != M["indic_format"]]
order_model        = [M["indic_order"]]
INSERT_PAGE_BREAKS = False

OUT_DIR    = Path("compare_out")
METHOD_DIR = OUT_DIR / "infinite_ocr"
CROP_DIR   = METHOD_DIR / "crops"
PAGE_DIR   = METHOD_DIR / "pages"
for d in (OUT_DIR, METHOD_DIR, CROP_DIR, PAGE_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ── CREDS (only OpenRouter + Bodhan) ──
BODHAN_KEY     = os.environ.get("BODHAN_OCR_API_KEY")
OPENROUTER_KEY = os.environ.get("OPENROUTER_API_KEY")
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
BODHAN_URL     = "https://api.bodhan.ai/v1/chat/completions"

PRICES_USD_PER_M = {                      # (input, output) USD per 1M tokens — used only if OpenRouter omits cost
    "google/gemini-2.5-flash-lite":   (0.10, 0.40),
    "google/gemini-2.5-flash":        (0.30, 2.50),
    "google/gemini-2.5-pro":          (1.25, 10.0),
    "qwen/qwen3-vl-30b-a3b-instruct": (0.10, 0.40),
    "qwen/qwen3.5-9b":                (0.10, 0.40),
    "nvidia/nemotron-3.5-super-vl":   (0.10, 0.40),
}
BODHAN_USD_PER_PAGE = 0.0024

# ── open PDF ──
doc   = pymupdf.open(PDF_PATH)
stem  = PDF_PATH.stem
total = len(doc)
page_indices = list(range(total)) if PAGES == -1 else [p for p in PAGES if 0 <= p < total]
if not page_indices:
    raise ValueError(f"no valid pages in PAGES={PAGES} for a {total}-page PDF")
print(f"PDF: {PDF_PATH.name} ({total} pages) · processing {len(page_indices)} page(s)")
print("models:", json.dumps(M, indent=1))


PDF: tmax.pdf (20 pages) · processing 20 page(s)
models: {
 "classify": "qwen/qwen3-vl-30b-a3b-instruct",
 "digital_order": "google/gemini-2.5-flash",
 "single_call": "google/gemini-2.5-flash",
 "indic_order": "google/gemini-2.5-flash",
 "indic_format": "google/gemini-2.5-flash",
 "ocr_fallback": "google/gemini-2.5-flash"
}


In [ ]:
# ─── CELL 2: metrics instrumentation (wraps requests.post) ────────────────
# Every API call to Bodhan / OpenRouter is logged with model, tokens, latency, cost.
# Call set_stage(method, stage) before each pipeline step so calls are attributed.

CALLS = []                       # one dict per HTTP call
_CTX  = {"method": "-", "stage": "-"}
_stage_t0 = {"method": "-", "stage": "-", "t": None}

def set_stage(method, stage):
    """Tag every HTTP call until the next set_stage(...) with (method, stage)."""
    global _stage_t0
    if _stage_t0["t"] is not None:
        CALLS.append({"method": _stage_t0["method"], "stage": _stage_t0["stage"],
                      "engine": "_wallclock_", "ok": True,
                      "tok_in": 0, "tok_out": 0, "cost": 0.0,
                      "seconds": time.perf_counter() - _stage_t0["t"]})
    _stage_t0.update(method=method, stage=stage, t=time.perf_counter())
    _CTX.update(method=method, stage=stage)

def reset_metrics():
    CALLS.clear(); _CTX.update(method="-", stage="-")
    _stage_t0.update(method="-", stage="-", t=None)
    print("[metrics] cleared")

def _extract(url, body_json, resp_json, dt, ok):
    model = "?"
    if isinstance(body_json, dict):
        model = body_json.get("model", model)
    if isinstance(resp_json, dict):
        model = resp_json.get("model") or model
    if model == "?" and "bodhan" in url.lower():
        model = "bodhan/indic-ocr"
    in_tok, out_tok, cost = 0, 0, None
    if isinstance(resp_json, dict):
        u = resp_json.get("usage") or {}
        in_tok  = u.get("prompt_tokens",     0) or 0
        out_tok = u.get("completion_tokens", 0) or 0
        if u.get("cost") is not None:
            cost = float(u["cost"])
    if cost is None:
        for key, (pin, pout) in PRICES_USD_PER_M.items():
            if key in model.lower():
                cost = (in_tok * pin + out_tok * pout) / 1_000_000
                break
    if cost is None and "bodhan" in model.lower() and BODHAN_USD_PER_PAGE is not None:
        cost = float(BODHAN_USD_PER_PAGE)
    return {"method": _CTX["method"], "stage": _CTX["stage"], "engine": model,
            "ok": ok, "tok_in": in_tok, "tok_out": out_tok,
            "cost": cost, "seconds": dt}

# install wrapper (idempotent — safe to re-run this cell)
if not getattr(requests.post, "_instrumented", False):
    _orig_post = requests.post

    def _tracked_post(url, *args, **kwargs):
        # ask OpenRouter to include cost in the usage block
        if "openrouter.ai" in url and isinstance(kwargs.get("json"), dict):
            _b = {**kwargs["json"], "usage": {"include": True}}
            if GEMINI_THINK_TOKENS is not None and "gemini" in str(_b.get("model", "")).lower() and "reasoning" not in _b:
                _b["reasoning"] = {"max_tokens": int(GEMINI_THINK_TOKENS)}
            kwargs["json"] = _b
        t0 = time.perf_counter()
        body = kwargs.get("json")
        try:
            resp = _orig_post(url, *args, **kwargs)
        except Exception as e:
            dt = time.perf_counter() - t0
            CALLS.append(_extract(url, body, None, dt, ok=False))
            raise
        dt = time.perf_counter() - t0
        try:    data = resp.json()
        except Exception: data = None
        CALLS.append(_extract(url, body, data, dt, ok=resp.ok))
        return resp

    _tracked_post._instrumented = True
    requests.post = _tracked_post
    print("[metrics] requests.post wrapped — call set_stage(method, stage) before each step")
else:
    print("[metrics] already wrapped — this cell is idempotent")


[metrics] already wrapped — this cell is idempotent


In [ ]:
# ─── CELL 3: shared helpers ──────────────────────────────────────────────
# Preprocessing + sanitisers + cross-page continuation detector + font helper.

# ── preprocess: three modes, same signature as the previous notebook ──────
def _illumination_normalize(bgr, sigma=35):
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    bg = cv2.GaussianBlur(gray, (0, 0), sigma)
    norm = cv2.divide(gray, bg, scale=192).clip(0, 255).astype(np.uint8)
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    lab[..., 0] = norm
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

def _clahe(bgr, clip=3.0):
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    lab[..., 0] = cv2.createCLAHE(clipLimit=clip, tileGridSize=(8, 8)).apply(lab[..., 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

def render_page(pno):
    pix = doc[pno].get_pixmap(dpi=DPI, alpha=False)
    return cv2.imdecode(np.frombuffer(pix.tobytes("png"), np.uint8), cv2.IMREAD_COLOR)

def preprocess_page(pno, method="original"):
    bgr = render_page(pno)
    if method == "original":
        return bgr
    if method == "gray":
        gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
        return cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
    if method == "clahe_mild":
        lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        l = cv2.createCLAHE(clipLimit=1.5, tileGridSize=(8, 8)).apply(l)
        return cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2BGR)
    raise ValueError(f"Unknown preprocessing method: {method}")


# ── sanitisers: strip HTML tags and LaTeX math from OCR output ────────────
_HTML_TAG_RE = re.compile(r"<[^>]+>")

def _strip_html(text):
    if not text: return ""
    s = text
    s = re.sub(r"<br\s*/?>",   "\n", s, flags=re.I)
    s = re.sub(r"</tr\s*>",    "\n", s, flags=re.I)
    s = re.sub(r"</t[dh]\s*>", "\n", s, flags=re.I)
    s = _HTML_TAG_RE.sub("", s)
    s = unescape(s)
    s = re.sub(r"[ \t]+\n", "\n", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()

_SUPER = str.maketrans("0123456789+-=()n", "⁰¹²³⁴⁵⁶⁷⁸⁹⁺⁻⁼⁽⁾ⁿ")
_SUB   = str.maketrans("0123456789+-=()",  "₀₁₂₃₄₅₆₇₈₉₊₋₌₍₎")
_LATEX_CMDS = {
    r"\alpha":"α", r"\beta":"β", r"\gamma":"γ", r"\delta":"δ", r"\epsilon":"ε",
    r"\theta":"θ", r"\lambda":"λ", r"\mu":"μ", r"\pi":"π", r"\rho":"ρ",
    r"\sigma":"σ", r"\tau":"τ", r"\phi":"φ", r"\omega":"ω",
    r"\Gamma":"Γ", r"\Delta":"Δ", r"\Theta":"Θ", r"\Lambda":"Λ",
    r"\Pi":"Π", r"\Sigma":"Σ", r"\Phi":"Φ", r"\Omega":"Ω",
    r"\infty":"∞", r"\times":"×", r"\cdot":"·", r"\div":"÷", r"\pm":"±",
    r"\leq":"≤", r"\geq":"≥", r"\neq":"≠", r"\approx":"≈",
    r"\rightarrow":"→", r"\partial":"∂", r"\sum":"∑", r"\int":"∫", r"\sqrt":"√",
    r"\degree":"°", r"\circ":"°",
    r"\overline":"", r"\bar":"", r"\hat":"", r"\vec":"",
    r"\text":"", r"\mathrm":"", r"\mathbf":"", r"\mathit":"",
    r"\left":"", r"\right":"",
    r"\,":" ", r"\;":" ", r"\!":"", r"\ ":" ",
}

def _sup(b):
    return b.translate(_SUPER) if b and all(c in "0123456789+-=()n" for c in b) else f"^({b})"
def _sub(b):
    return b.translate(_SUB)   if b and all(c in "0123456789+-=()"  for c in b) else f"_({b})"

def _flatten_latex(s):
    s = re.sub(r"(\d)\\frac", r"\1 \\frac", s)
    for k, v in _LATEX_CMDS.items(): s = s.replace(k, v)
    s = re.sub(r"\\frac\s*\{([^{}]*)\}\s*\{([^{}]*)\}", r"\1/\2", s)
    s = re.sub(r"\^\{([^{}]*)\}",  lambda m: _sup(m.group(1)), s)
    s = re.sub(r"\^([A-Za-z0-9])", lambda m: _sup(m.group(1)), s)
    s = re.sub(r"_\{([^{}]*)\}",   lambda m: _sub(m.group(1)), s)
    s = re.sub(r"_([A-Za-z0-9])",  lambda m: _sub(m.group(1)), s)
    s = re.sub(r"\\[A-Za-z]+", "", s)
    return s.replace("{", "").replace("}", "").strip()

def _strip_latex(text):
    if not text: return ""
    text = re.sub(r"\$\$(.+?)\$\$", lambda m: _flatten_latex(m.group(1)), text, flags=re.S)
    text = re.sub(r"\$(.+?)\$",     lambda m: _flatten_latex(m.group(1)), text, flags=re.S)
    return text

def sanitize(text):
    return _strip_latex(_strip_html(text or ""))


# ── cross-page continuation detector (kept as a safety net) ───────────────
_TERMINAL_PUNCT_RE = re.compile(r"[.!?।॥:;\"')\]\}]\s*$")
_FURNITURE_RE = re.compile(
    r"^\s*("
    r"scanned\s+(by|with)\s+.+"
    r"|page\s+\d+(\s+of\s+\d+)?"
    r"|\[?\s*p\.?\s*t\.?\s*o\.?\s*\]?"
    r"|\d+-\d+-\d+"
    r"|\[\s*\]"
    r")\s*$", re.I)

_BODY_TYPES = {"paragraph", "list_item"}

def _is_real_body(region):
    if region.get("type") not in _BODY_TYPES: return False
    t = (region.get("text") or "").strip()
    if not t: return False
    return not _FURNITURE_RE.match(t)


# ── JSON parse helper (shared across stages) ──────────────────────────────
def _parse_json_object(text):
    s = (text or "").strip()
    if s.startswith("```"):
        lines = s.splitlines()
        if lines and lines[0].startswith("```"): lines = lines[1:]
        if lines and lines[-1].strip() == "```": lines = lines[:-1]
        s = "\n".join(lines)
    try: return json.loads(s)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", s, re.S)
        return json.loads(m.group(0)) if m else {}


# ── font helper (shared with DOCX render in Stage 3) ──────────────────────
_FONT_NAME = "Nirmala UI"

def _apply_font(run, name=_FONT_NAME):
    run.font.name = name
    try:
        from docx.oxml.ns import qn
        from docx.oxml import OxmlElement
        rPr = run._element.get_or_add_rPr()
        rFonts = rPr.find(qn("w:rFonts"))
        if rFonts is None:
            rFonts = OxmlElement("w:rFonts"); rPr.insert(0, rFonts)
        for a in ("w:ascii", "w:hAnsi", "w:eastAsia", "w:cs"):
            rFonts.set(qn(a), name)
    except Exception:
        pass


# ── small preview helpers used by diagnostic prints ───────────────────────
def _preview(t, n=70):
    t = (t or "").replace("\n", " ⏎ ")
    return t[:n]

def _show_block(tag, b, label=None):
    bb = b.get("bbox_xyxy")
    bb_s = "null" if bb is None else f"[{int(bb[0])},{int(bb[1])},{int(bb[2])},{int(bb[3])}]"
    lab = label if label is not None else b.get("_label", "?")
    print(f"    {tag:<10} {lab:>5}  type={b.get('type','?'):<14} "
          f"bbox={bb_s:<28} text={_preview(b.get('text'))!r}")

print("shared helpers loaded")


shared helpers loaded


In [ ]:
# ─── CELL: tag grammar shared by every route (sliced from the combined prompt) ───
TAG_GRAMMAR = r"""Units: font size in points; indents in inches; spacing in points. Leave an attribute out when it is the default or when you cannot judge it.

----------------------------------------------------------------
<P …>text</P>
----------------------------------------------------------------
Every paragraph is a P. Attributes:
  align         left | center | right | justify   (default left). ALWAYS set align when visibly centered, right-aligned or justified.
  size          font size in points. Ordinary body text ≈ 11. Estimate by comparing letter height with ordinary body text on the SAME page (main title ≈ 16–22, section header ≈ 13–15, body ≈ 11, small print / notes ≈ 8–10). Leave out for body size.
  bold | italic | underline   "true" only when the WHOLE paragraph is visibly so.
  font          only if clearly identifiable (e.g. "Times New Roman", "Arial"); never for Indic scripts.
  color         "#RRGGBB" only for clearly coloured text.
  heading       1 | 2 | 3 for the document / institution title, section headers, and small sub-labels: a line that is visibly larger, bolder or centered and stands apart from body text.
  indent_left, indent_first (negative = hanging), space_before, space_after, line_spacing (1.0 = single): only when clearly different from normal.

----------------------------------------------------------------
<STYLE bold= italic= underline= strike= size= font= color="#RRGGBB" highlight= superscript= subscript=>text</STYLE>
----------------------------------------------------------------
Inline character formatting for PART of a paragraph. Use the smallest span.

----------------------------------------------------------------
<BR/>
----------------------------------------------------------------
Intentional line break inside a paragraph.

----------------------------------------------------------------
<BLANK type="underscore|dots|line|space" length="N"/>
----------------------------------------------------------------
A fill-in-the-blank run, at exactly the same spot. length = the number of characters in the OCR run (count them). Use ONLY where the OCR contains such a run (3+ underscores / dots / dashes). Never add a blank the OCR text does not have; never fill a blank with an answer.

----------------------------------------------------------------
<LIST type="bullet|number|alpha|roman|none"><ITEM level="0">…</ITEM></LIST>
----------------------------------------------------------------
Use <LIST> only when the OCR text shows a vertical column of items with a leading marker on each item.

CRITICAL — label preservation: inside each <ITEM>, the first characters of the OCR line are the item's label, and they are CONTENT, not formatting. You MUST copy them verbatim into the <ITEM> text, together with the punctuation and the single space that follows. The downstream builder does NOT auto-number list items; whatever label is missing from your <ITEM> text is missing from the final document.

What counts as a label (script-independent): any short run of 1–3 characters at the start of the OCR line, followed by a punctuation mark (one of ".", ")", "(", "-", ":", "/", "]", "}") and whitespace, AND the same shape repeats on the next sibling items in the list. Script does not matter — the label can be a digit, a Latin letter, a roman numeral, a single letter or ligature of any Indic / CJK / Cyrillic / Arabic script, a circled number, or a pure bullet ("•", "·", "-", "*", "–", "—", "◦", "▪", "➤"). Keep the entire label — its character(s), its punctuation and the space after it — verbatim in the <ITEM> text.

If a line in the OCR has no leading marker and you cannot be certain the group is a list, emit <P> for each line instead of <LIST>. Faithful content always beats a prettier layout.

----------------------------------------------------------------
<QUESTION> … </QUESTION>   optional wrapper around a numbered item; a P for the stem, then
<OPTIONS layout="vertical|grid" columns="2"><OPTION>(a) …</OPTION> …</OPTIONS>
----------------------------------------------------------------
Same label-preservation rule as <LIST>: the option marker is CONTENT, whatever script it is in. Copy it verbatim into the <OPTION> text with its following punctuation and space. Keep every option in the original order. vertical = one option per line. grid = options arranged in N columns on the page.

----------------------------------------------------------------
<TABLE border="1" align="left|center|right"><ROW header="true"><CELL align= bold= size= colspan="2">text</CELL></ROW></TABLE>
----------------------------------------------------------------
CRITICAL TABLE RULE: You MUST convert all tabular data (whether in Markdown format with `|` pipes or HTML markup) into this exact XML structure. 
- NEVER output raw Markdown pipe (`|`) characters in the final text.
- Always include border="1" so the downstream document builder applies proper grid lining.
- Keep cell text exactly as in the OCR. Ensure every row covers the same number of columns by using the colspan attribute where necessary.

----------------------------------------------------------------
Symbols
----------------------------------------------------------------
Keep all symbols, maths and Unicode exactly. Use <STYLE superscript=…> and <STYLE subscript=…> for raised / lowered characters. Never replace a formula by words.
"""


In [ ]:
# ─── CELL: shared utils for the non-Bodhan routes ──────────────────────────
import threading
_DOC_LOCK = threading.Lock()                 # PyMuPDF Document is not thread-safe: serialise page access
_render_page_unsafe = render_page
def render_page(pno):
    with _DOC_LOCK: return _render_page_unsafe(pno)
_PAGE_WH = {p: (doc[p].rect.width, doc[p].rect.height) for p in page_indices}

def _x(s):                       # escape text for the tag markup
    return (s or "").replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")

def _img_uri(bgr, max_side=1600, q=85):
    h, w = bgr.shape[:2]; s = min(1.0, max_side / max(h, w))
    if s < 1: bgr = cv2.resize(bgr, (int(w * s), int(h * s)), interpolation=cv2.INTER_AREA)
    return "data:image/jpeg;base64," + base64.b64encode(cv2.imencode(".jpg", bgr, [cv2.IMWRITE_JPEG_QUALITY, q])[1].tobytes()).decode()

def _or_chat(model, content, json_mode=False, timeout=240, tries=5, max_tokens=None):
    import random
    if not OPENROUTER_KEY: raise RuntimeError("OPENROUTER_API_KEY is not set")
    body = {"model": model, "temperature": 0.0, "messages": [{"role": "user", "content": content}]}
    if json_mode: body["response_format"] = {"type": "json_object"}
    if max_tokens: body["max_tokens"] = max_tokens
    last = None
    for a in range(tries):
        wait = min(40, 3 * 2 ** a) + random.uniform(0, 2)           # 3s, 6s, 12s, 24s, 40s (+jitter)
        try:
            r = requests.post(OPENROUTER_URL, headers={"Authorization": f"Bearer {OPENROUTER_KEY}", "Content-Type": "application/json"},
                              json=body, timeout=timeout)
            if r.status_code in (429, 500, 502, 503, 504):
                ra = r.headers.get("Retry-After")
                if ra and ra.isdigit(): wait = max(wait, min(60, int(ra)))
                last = RuntimeError(f"HTTP {r.status_code} from {model}"); time.sleep(wait); continue
            if r.status_code == 402: raise RuntimeError("HTTP 402: OpenRouter credits exhausted")
            r.raise_for_status(); data = r.json()
            if "error" in data: last = RuntimeError(str(data["error"])[:200]); time.sleep(wait); continue
            return data["choices"][0]["message"]["content"] or ""
        except requests.RequestException as e:
            last = e; time.sleep(wait)
    raise RuntimeError(f"openrouter failed after {tries} tries: {last}")

def _json_from(text):
    s = (text or "").strip()
    s = re.sub(r"^```(?:json)?|```$", "", s, flags=re.M).strip()
    try: return json.loads(s)
    except Exception:
        m = re.search(r"\{.*\}", s, re.S)
        return json.loads(m.group(0)) if m else {}

_DROP_RE = re.compile(r"<DROP>.*?</DROP>", re.S)
def _strip_drop(markup): return _DROP_RE.sub("", markup or "").strip()

def _mkblock(bid, page, bbox_norm, btype, markup="", picture_path=None, wh=None):
    W, H = wh or (0, 0)
    bb = [round(bbox_norm[0]*W, 1), round(bbox_norm[1]*H, 1), round(bbox_norm[2]*W, 1), round(bbox_norm[3]*H, 1)] if bbox_norm else None
    return {"block_id": bid, "page": page, "bbox_xyxy": bb, "bbox_norm": bbox_norm, "bodhan_type": btype,
            "is_picture": bool(picture_path), "image_path": picture_path, "formatted_text": markup}


## Shared DOCX builder (tag parser + renderer, from your proven notebook)

In [ ]:
# ── tag-parser prerequisites (copied from Stage 2 cell so the builder works on every route) ──
STRIP_LITERAL_BACKSLASH_N = True
_KNOWN_TAGS = "P|STYLE|BR|BLANK|LIST|ITEM|QUESTION|OPTIONS|OPTION|TABLE|ROW|CELL"
_TAG_RE = re.compile(
    r"<(?P<close>/?)(?P<name>%s)(?P<attrs>(?:\s+[A-Za-z_][\w-]*\s*=\s*(?:\"[^\"]*\"|'[^']*'|[^\s<>/\"']+))*)\s*(?P<self>/?)>"
    % _KNOWN_TAGS, re.I)
_ATTR_RE = re.compile(r"""([A-Za-z_][\w-]*)\s*=\s*("[^"]*"|'[^']*'|[^\s<>/"']+)""")
_VOID_TAGS = {"BR", "BLANK"}
def _parse_attrs(s):
    return {k.lower(): v.strip("\"'") for k, v in _ATTR_RE.findall(s or "")}

# ─── CELL 6: Stage 3 — tag parser + DOCX render + sidecar + stats ────────

# ── tag parser ──────────────────────────────────────────────────────────
class _Node:
    __slots__ = ("tag", "attrs", "children")
    def __init__(self, tag, attrs=None):
        self.tag, self.attrs, self.children = tag, attrs or {}, []

_XML_BAD_RE = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\ufffe\uffff]")

def _parse_markup(text):
    """Tolerant tag-tree parser. Unknown '<…>' stays literal text."""
    root = _Node("ROOT")
    stack, pos = [root], 0
    for m in _TAG_RE.finditer(text):
        if m.start() > pos:
            stack[-1].children.append(_html.unescape(text[pos:m.start()]))
        pos = m.end()
        name = m.group("name").upper()
        if m.group("close"):
            for k in range(len(stack) - 1, 0, -1):
                if stack[k].tag == name:
                    del stack[k:]
                    break
            continue
        node = _Node(name, _parse_attrs(m.group("attrs")))
        stack[-1].children.append(node)
        if name in _VOID_TAGS or m.group("self"):
            continue
        stack.append(node)
    if pos < len(text):
        stack[-1].children.append(_html.unescape(text[pos:]))
    return root


# ── attribute helpers ───────────────────────────────────────────────────
_ALIGN = {"left": WD_ALIGN_PARAGRAPH.LEFT, "center": WD_ALIGN_PARAGRAPH.CENTER,
          "centre": WD_ALIGN_PARAGRAPH.CENTER, "right": WD_ALIGN_PARAGRAPH.RIGHT,
          "justify": WD_ALIGN_PARAGRAPH.JUSTIFY}
_TBL_ALIGN = {"left": WD_TABLE_ALIGNMENT.LEFT, "center": WD_TABLE_ALIGNMENT.CENTER,
              "right": WD_TABLE_ALIGNMENT.RIGHT}
_HL = {"yellow": "YELLOW", "green": "BRIGHT_GREEN", "cyan": "TURQUOISE", "turquoise": "TURQUOISE",
       "pink": "PINK", "red": "RED", "blue": "BLUE", "gray": "GRAY_25", "grey": "GRAY_25"}

def _truthy(v): return str(v).strip().lower() in ("1", "true", "yes", "y", "on")

def _num(v):
    if v is None: return None
    m = re.search(r"-?\d+(?:\.\d+)?", str(v))
    return float(m.group()) if m else None

def _style_from_attrs(attrs, base):
    st = dict(base)
    for k in ("bold", "italic", "underline", "strike", "superscript", "subscript"):
        if k in attrs: st[k] = _truthy(attrs[k])
    s = _num(attrs.get("size"))
    if s: st["size"] = max(5.0, min(48.0, s))
    if attrs.get("font"):      st["font"] = attrs["font"].strip()
    if attrs.get("color"):     st["color"] = attrs["color"].strip()
    if attrs.get("highlight"): st["highlight"] = attrs["highlight"].strip().lower()
    return st

def _apply_run_style(r, st):
    if "bold" in st:        r.bold = st["bold"]
    if "italic" in st:      r.italic = st["italic"]
    if "underline" in st:   r.underline = st["underline"]
    if st.get("strike"):      r.font.strike = True
    if st.get("superscript"): r.font.superscript = True
    if st.get("subscript"):   r.font.subscript = True
    if st.get("size"):  r.font.size = Pt(st["size"])
    if st.get("font"):  r.font.name = st["font"]
    c = (st.get("color") or "").lstrip("#")
    if re.fullmatch(r"[0-9a-fA-F]{6}", c):
        r.font.color.rgb = RGBColor.from_string(c.upper())
    hl = _HL.get(st.get("highlight", ""))
    if hl and hasattr(WD_COLOR_INDEX, hl):
        r.font.highlight_color = getattr(WD_COLOR_INDEX, hl)

def _blank_text(attrs):
    t = (attrs.get("type") or "underscore").lower()
    n = int(_num(attrs.get("length")) or 10)
    n = max(1, min(120, n))
    ch = {"dots": ".", "space": "\u00a0", "line": "_", "underscore": "_"}.get(t, "_")
    return ch * n


# ── inline flattening + emit ────────────────────────────────────────────
def _flatten_inline(nodes, style, out):
    for n in nodes:
        if isinstance(n, str):
            t = _XML_BAD_RE.sub("", n)
            if STRIP_LITERAL_BACKSLASH_N: t = t.replace("\\n", " ")
            t = re.sub(r"\s+", " ", t)
            if t: out.append(("text", t, style))
        elif n.tag == "STYLE":
            _flatten_inline(n.children, _style_from_attrs(n.attrs, style), out)
        elif n.tag == "BR":
            out.append(("br", None, style))
        elif n.tag == "BLANK":
            out.append(("blank", n.attrs, style))
        else:
            _flatten_inline(n.children, style, out)
    return out

def _trim_segments(segs):
    segs = list(segs)
    for i, (k, v, st) in enumerate(segs):
        if k != "text": continue
        if i == 0 or segs[i - 1][0] == "br":                 v = v.lstrip()
        if i == len(segs) - 1 or segs[i + 1][0] == "br":     v = v.rstrip()
        segs[i] = (k, v, st)
    return [s for s in segs if not (s[0] == "text" and not s[1])]

def _emit_segments(p, segs):
    for kind, val, st in segs:
        if kind == "br":
            p.add_run().add_break(); continue
        r = p.add_run(_blank_text(val) if kind == "blank" else val)
        _apply_font(r)
        _apply_run_style(r, st)

def _apply_para_format(p, attrs, hanging=None, left_default=None):
    al = _ALIGN.get(str(attrs.get("align", "")).lower())
    if al is not None: p.alignment = al
    pf = p.paragraph_format
    li, fi = _num(attrs.get("indent_left")), _num(attrs.get("indent_first"))
    if li is None: li = left_default
    if fi is None: fi = hanging
    if li is not None: pf.left_indent = Inches(max(-3, min(8, li)))
    if fi is not None: pf.first_line_indent = Inches(max(-3, min(8, fi)))
    sb, sa, ls = _num(attrs.get("space_before")), _num(attrs.get("space_after")), _num(attrs.get("line_spacing"))
    if sb is not None: pf.space_before = Pt(max(0, min(72, sb)))
    if sa is not None: pf.space_after  = Pt(max(0, min(72, sa)))
    if ls is not None and 0.5 <= ls <= 3: pf.line_spacing = ls

def _heading_level(attrs):
    v = str(attrs.get("heading", "")).lower().lstrip("h")
    return int(v) if v in ("1", "2", "3") else None

def _spacer(docx):
    p = docx.add_paragraph("")
    pf = p.paragraph_format
    pf.space_before = Pt(0); pf.space_after = Pt(0)
    pf.line_spacing_rule = WD_LINE_SPACING.EXACTLY
    pf.line_spacing = Pt(4)
    return p


# ── element renderers ──────────────────────────────────────────────────
def _render_p(docx, node, hanging=None, left_default=None):
    attrs = node.attrs
    segs = _trim_segments(_flatten_inline(node.children, _style_from_attrs(attrs, {}), []))
    if not segs: return None
    lvl = _heading_level(attrs)
    if lvl:
        p = docx.add_heading("", level=lvl)
        segs = [(k, v, {**{"color": "000000", "bold": False}, **st}) for k, v, st in segs]
    else:
        p = docx.add_paragraph()
    _apply_para_format(p, attrs, hanging, left_default)
    _emit_segments(p, segs)
    return p

def _fill_cell(cell, node, header=False):
    base = _style_from_attrs(node.attrs, {"bold": True} if header else {})
    paras = [c for c in node.children if isinstance(c, _Node) and c.tag in ("P", "ITEM", "OPTION")]
    if paras:
        for i, pn in enumerate(paras):
            p = cell.paragraphs[0] if i == 0 else cell.add_paragraph()
            segs = _trim_segments(_flatten_inline(pn.children, _style_from_attrs(pn.attrs, base), []))
            _apply_para_format(p, pn.attrs)
            _emit_segments(p, segs)
    else:
        p = cell.paragraphs[0]
        segs = _trim_segments(_flatten_inline(node.children, base, []))
        _apply_para_format(p, node.attrs)
        _emit_segments(p, segs)

def _render_table(docx, node, produced):
    rows = [c for c in node.children if isinstance(c, _Node) and c.tag == "ROW"]
    if not rows:
        _render_nodes(docx, node.children, produced); return
    grid = []
    for r in rows:
        cells = [c for c in r.children if isinstance(c, _Node) and c.tag == "CELL"]
        stray = "".join(c for c in r.children if isinstance(c, str)).strip()
        if stray:
            pseudo = _Node("CELL"); pseudo.children = [stray]; cells.append(pseudo)
        grid.append((r, cells))
    spans = [[max(1, min(12, int(_num(c.attrs.get("colspan")) or 1))) for c in cells] for _, cells in grid]
    ncols = max(1, max(sum(s) for s in spans))
    tbl = docx.add_table(rows=len(grid), cols=ncols)
    if str(node.attrs.get("border", "1")).lower() not in ("0", "false", "none", "no"):
        tbl.style = "Table Grid"
    ta = _TBL_ALIGN.get(str(node.attrs.get("align", "")).lower())
    if ta is not None: tbl.alignment = ta
    for ri, (row, cells) in enumerate(grid):
        header = _truthy(row.attrs.get("header", "false"))
        ci = 0
        for cn, span in zip(cells, spans[ri]):
            if ci >= ncols: break
            cell = tbl.cell(ri, ci)
            if span > 1:
                cell = cell.merge(tbl.cell(ri, min(ncols - 1, ci + span - 1)))
            _fill_cell(cell, cn, header)
            ci += span
    _spacer(docx); produced.append(len(docx.paragraphs) - 1)

def _render_options(docx, node, produced):
    ok_tags = ("OPTION", "P", "ITEM")
    clean = all((isinstance(c, _Node) and c.tag in ok_tags) or (isinstance(c, str) and not c.strip())
                for c in node.children)
    opts = [c for c in node.children if isinstance(c, _Node)]
    if not clean or not opts:
        _render_nodes(docx, node.children, produced); return
    layout = str(node.attrs.get("layout", "vertical")).lower()
    if layout in ("grid", "horizontal", "inline") and len(opts) > 1:
        default_cols = 2 if layout == "grid" else len(opts)
        ncols = max(1, min(6, int(_num(node.attrs.get("columns")) or default_cols)))
        nrows = math.ceil(len(opts) / ncols)
        tbl = docx.add_table(rows=nrows, cols=ncols)
        for i, o in enumerate(opts):
            _fill_cell(tbl.cell(i // ncols, i % ncols), o)
        _spacer(docx); produced.append(len(docx.paragraphs) - 1)
    else:
        for o in opts:
            if _render_p(docx, o, None, 0.4) is not None:
                produced.append(len(docx.paragraphs) - 1)

def _render_nodes(docx, nodes, produced):
    buf = []
    def flush():
        if not buf: return
        pseudo = _Node("P"); pseudo.children = list(buf); buf.clear()
        if _render_p(docx, pseudo) is not None:
            produced.append(len(docx.paragraphs) - 1)
    for n in nodes:
        if isinstance(n, str):
            flush()
            t = n.replace("\\n", "\n") if STRIP_LITERAL_BACKSLASH_N else n
            for line in t.split("\n"):
                if line.strip():
                    pseudo = _Node("P"); pseudo.children = [line]
                    if _render_p(docx, pseudo) is not None:
                        produced.append(len(docx.paragraphs) - 1)
            continue
        if n.tag in ("STYLE", "BR", "BLANK"):
            buf.append(n); continue
        flush()
        if n.tag in ("P", "ITEM", "OPTION"):
            if _render_p(docx, n) is not None: produced.append(len(docx.paragraphs) - 1)
        elif n.tag == "LIST":
            ltype = str(n.attrs.get("type", "none")).lower()
            for ch in n.children:
                if isinstance(ch, _Node) and ch.tag in ("ITEM", "P"):
                    lvl = max(0, min(4, int(_num(ch.attrs.get("level")) or 0)))
                    ld, hg = (0.3 * lvl, None) if ltype == "none" else (0.3 * (lvl + 1), -0.25)
                    if _render_p(docx, ch, hg, ld) is not None:
                        produced.append(len(docx.paragraphs) - 1)
                else:
                    _render_nodes(docx, [ch], produced)
        elif n.tag == "OPTIONS":
            _render_options(docx, n, produced)
        elif n.tag == "TABLE":
            _render_table(docx, n, produced)
        else:
            _render_nodes(docx, n.children, produced)
    flush()

def _render_formatted_text(docx, text):
    produced = []
    try:
        _render_nodes(docx, _parse_markup(text).children, produced)
    except Exception as e:
        print(f"    ⚠ markup render failed ({type(e).__name__}: {str(e)[:120]}) — writing plain text")
        plain = _html.unescape(_TAG_RE.sub(" ", text))
        for line in plain.split("\n"):
            if line.strip():
                docx.add_paragraph(_XML_BAD_RE.sub("", line.strip()))
                produced.append(len(docx.paragraphs) - 1)
    return produced


# ── build the DOCX in reading order (= order of ordered_blocks) ─────────
def build_docx_with_blocks(blocks, out_path):
    docx = Document()
    docx.styles["Normal"].font.name = _FONT_NAME
    docx.styles["Normal"].font.size = Pt(11)

    block_meta, last_page = {}, None
    for b in blocks:
        bid = b["block_id"]
        base = {"page": b["page"], "bbox_xyxy": b.get("bbox_xyxy"), "bodhan_type": b["bodhan_type"]}

        if INSERT_PAGE_BREAKS and last_page is not None and b["page"] != last_page:
            docx.add_page_break()
        last_page = b["page"]

        if b.get("is_picture") and b.get("image_path"):
            try:
                docx.add_picture(b["image_path"], width=Inches(5.5))
                idx = len(docx.paragraphs) - 1
                block_meta[bid] = {**base, "rendered": True, "docx_paragraph_range": [idx, idx]}
            except Exception as e:
                docx.add_paragraph(f"[picture failed: {e}]")
                block_meta[bid] = {**base, "rendered": False, "reason": f"picture_error: {e}",
                                   "docx_paragraph_range": None}
            continue

        text = (b.get("formatted_text") or "").strip()
        if not text:
            block_meta[bid] = {**base, "rendered": False, "reason": "empty_text_or_block",
                               "docx_paragraph_range": None}
            continue

        produced = _render_formatted_text(docx, text)
        if produced:
            block_meta[bid] = {**base, "rendered": True,
                               "docx_paragraph_range": [produced[0], produced[-1]]}
        else:
            block_meta[bid] = {**base, "rendered": False, "reason": "no_paragraphs_emitted",
                               "docx_paragraph_range": None}

    docx.save(str(out_path))
    return str(out_path), block_meta


## Router

In [ ]:
# ─── CELL: ROUTER — follows the tree ──────────────────────────────────────
DIGITAL_MIN_CHARS_PER_PAGE = 120     # median text chars/page to call a PDF "digital"
DIGITAL_MIN_PAGE_SHARE     = 0.6     # share of sampled pages that must have a text layer
ENGLISH_MIN_LATIN_SHARE    = 0.92    # share of letters that are plain Latin to call digital text "English"
SCRIPT_PROBE_PAGES         = 2       # scanned: pages sent to the cheap script probe
LATIN_HINDI_OK = {"latin", "english", "devanagari", "hindi", "digits", "numerals"}

def _digital_stats():
    pcs, lat = [], []
    for p in page_indices:
        t = doc[p].get_text()
        pcs.append(len(t.strip()))
        L = [ch for ch in t if ch.isalpha()]
        lat.append((sum(ch.isascii() for ch in L), len(L)))
    a, b = sum(x for x, _ in lat), sum(y for _, y in lat)
    share_txt = sum(c >= 30 for c in pcs) / len(pcs)
    return statistics.median(pcs), share_txt, (a / b if b else 0.0)

def _probe_scripts():
    if not OPENROUTER_KEY: return None
    # content = [{"type": "text", "text": 'List every writing script (not language) that appears as real text on these document pages, '
    #                                     'e.g. Latin, Devanagari, Telugu, Tamil, Kannada, Malayalam, Bengali, Gujarati, Gurmukhi, Odia, Arabic. '
    #                                     'Ignore tiny stamps/watermarks. Return ONLY JSON: {"scripts": ["..."]}'}]
    content = [{
        "type": "text", 
        "text": 'Look at these document pages. List every writing script (such as English, Hindi, Marathi, Telugu, Tamil, Kannada, etc.) found in the text. '
                'Ignore watermarks. You must respond with a JSON object containing a single key "scripts" whose value is a list of strings. Example: {"scripts": ["English", "telugu"]}'
    }]
    
    for p in page_indices[:SCRIPT_PROBE_PAGES]:
        content.append({"type": "image_url", "image_url": {"url": _img_uri(render_page(p), max_side=900, q=70)}})
    models = [M["classify"]] + [m for m in MODEL_OPTIONS["classify"] if m != M["classify"]]
    for m in models:
        try:
            return [x.strip().lower() for x in (_json_from(_or_chat(m, content, json_mode=True, max_tokens=200, tries=3)).get("scripts") or [])]
        except Exception as e:
            print(f"  script probe failed on {m}: {str(e)[:120]}")
    print("  script probe failed on every model -> defaulting to the Indic route (set FORCE_ROUTE to override)")
    return None

def decide_route():
    med, share, latin = _digital_stats()
    digital = med >= DIGITAL_MIN_CHARS_PER_PAGE and share >= DIGITAL_MIN_PAGE_SHARE
    info = {"median_chars_per_page": med, "pages_with_text": round(share, 2), "latin_share": round(latin, 3)}
    if digital:
        route = "digital_english" if latin >= ENGLISH_MIN_LATIN_SHARE else "single_gemini"
        info["kind"] = "digital · " + ("English" if route == "digital_english" else "non-English")
    else:
        scripts = _probe_scripts()
        info["scripts"] = scripts
        if scripts is None:
            route = "scanned_indic"; info["kind"] = "scanned · script probe unavailable → default Indic route"
        elif set(scripts) <= LATIN_HINDI_OK:
            route = "single_gemini"; info["kind"] = "scanned · English/Hindi"
        else:
            route = "scanned_indic"; info["kind"] = "scanned · Indic"
    return route, info

ROUTE, ROUTE_INFO = decide_route()
if FORCE_ROUTE: ROUTE = FORCE_ROUTE; ROUTE_INFO["forced"] = True
print("ROUTE =", ROUTE, "\n", json.dumps(ROUTE_INFO, indent=1))


ROUTE = digital_english 
 {
 "median_chars_per_page": 3575.5,
 "pages_with_text": 1.0,
 "latin_share": 1.0,
 "kind": "digital \u00b7 English"
}


## ROUTE 1 — digital English

In [ ]:
# ─── ROUTE 1: digital English → Docling Heron → ordering prompt → tags ────
DIGITAL_ORDER_PROMPT = """You get ONE page: the page image with every detected block boxed and labelled (B1, B2, ...), and a JSON list of those blocks (NOT in reading order). Return the reading order.
Fields: id (the label on the image), type, bbox [x0,y0,x1,y1] normalised 0-1 from top-left, text (long blocks: text_start + text_end; tables/figures/formulas carry no text).

RULES
1 LAYOUT. Look at the boxes. Split the page into horizontal ZONES (top to bottom) wherever a full-width block spans the page or the column count changes. Inside a multi-column zone read the LEFT column completely top to bottom, then the next column; never read across a row. Blocks side by side in one row (e.g. two images): left to right. Use bbox + image, not the JSON order.
2 PAIRING. A caption stays with its figure/table (figure then caption; a table caption on the side of the table where it sits). A title/section_header goes directly before the text it introduces. A formula stays where it sits in its column.
3 TEXT CONTINUITY (the only thing allowed to change the base order; judge from the text):
 a) If the first body text block of the page starts mid-sentence (fragment / lowercase), it continues the previous page: put it FIRST, before any figure, table or caption; figures/tables that moved behind it follow right after it.
 b) If the last text block of a column ends cut off and the first text block of the next column completes it, place that block right after it; figures/tables lying between go after the completed block.
 c) A missing final period alone is NOT a cut-off. If unsure, keep the base order.
4 FURNITURE. Footnotes after the body, then page_header, page_footer, page_number at the very end. Exception: if the last body block is cut off mid-sentence (continues next page), put the furniture just BEFORE it so it is last.
Every id exactly once, only ids from the list. Answer with ONLY this JSON, no explanation: {"order": ["B3","B1", ...]}"""

_HERON_MAP = {"text": "text", "paragraph": "text", "title": "title", "section_header": "section_header", "list_item": "list_item",
              "caption": "caption", "footnote": "footnote", "table": "table", "document_index": "table", "picture": "picture",
              "formula": "formula", "page_header": "page_header", "page_footer": "page_footer", "code": "text"}
_FURNITURE = {"page_header", "page_footer"}
_FIGURE    = {"picture", "formula"}

def heron_layout():
    """Docling (Heron layout model) -> {page_idx: [{'type','rect'(pt, top-left)}]}. OCR + table-structure off: text comes from the PDF text layer."""
    from docling.document_converter import DocumentConverter, PdfFormatOption
    from docling.datamodel.base_models import InputFormat
    from docling.datamodel.pipeline_options import PdfPipelineOptions
    opts = PdfPipelineOptions(); opts.do_ocr = False; opts.do_table_structure = False
    conv = DocumentConverter(format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=opts)})
    lo, hi = min(page_indices), max(page_indices)
    res = conv.convert(str(PDF_PATH), page_range=(lo + 1, hi + 1))
    out = {}
    for k, pg in enumerate(res.pages):
        pno = lo + k
        if pno not in page_indices: continue
        H = doc[pno].rect.height
        items = []
        for c in pg.predictions.layout.clusters:
            bb = c.bbox
            if "BOTTOM" in str(getattr(bb, "coord_origin", "")).upper(): bb = bb.to_top_left_origin(H)
            lab = getattr(c.label, "value", str(c.label)).lower()
            items.append({"type": _HERON_MAP.get(lab, "text"), "rect": pymupdf.Rect(bb.l, bb.t, bb.r, bb.b) & doc[pno].rect})
        items = [i for i in items if not i["rect"].is_empty]
        items.sort(key=lambda i: (round(i["rect"].y0), i["rect"].x0))       # neutral order — the LLM decides reading order
        out[pno] = items
    return out

def _hyphen_join(prev, nxt):
    return prev[:-1] + nxt if prev.endswith("-") and nxt[:1].islower() and len(prev) > 1 and prev[-2].isalpha() else prev + " " + nxt

def _read_region(page, rect):
    """-> (segments[(text,bold,italic,sup)], plain_text, dominant_size, nchars)"""
    segs, sizes, plain = [], [], ""
    for b in page.get_text("dict", clip=rect)["blocks"]:
        if b.get("type") != 0: continue
        for ln in b["lines"]:
            lt = "".join(s["text"] for s in ln["spans"])
            if not lt.strip(): continue
            first = True
            for s in ln["spans"]:
                t = s["text"]
                if not t: continue
                fl, fn = s["flags"], s["font"].lower()
                st = (bool(fl & 16) or "bold" in fn or "black" in fn, bool(fl & 2) or "italic" in fn or "oblique" in fn, bool(fl & 1))
                if first and segs:                                   # line join
                    if plain.endswith("-") and t[:1].islower() and len(plain) > 1 and plain[-2].isalpha():
                        plain = plain[:-1]; segs[-1] = (segs[-1][0][:-1],) + segs[-1][1:]
                    else:
                        t = " " + t.lstrip(); 
                first = False
                plain += t; sizes += [s["size"]] * len(t.strip())
                if segs and segs[-1][1:] == st: segs[-1] = (segs[-1][0] + t,) + st
                else: segs.append((t,) + st)
    return segs, plain.strip(), (statistics.median(sizes) if sizes else 0), len(sizes)

def _markup(btype, segs, size, body_size, rect, pw):
    nb = sum(len(t) for t, b, i, s in segs if b) / max(1, sum(len(t) for t, *_ in segs))
    ni = sum(len(t) for t, b, i, s in segs if i) / max(1, sum(len(t) for t, *_ in segs))
    pb, pi = nb >= 0.9, ni >= 0.9
    inner = ""
    for t, b, i, s in segs:
        a = []
        if b and not pb: a.append('bold="true"')
        if i and not pi: a.append('italic="true"')
        if s: a.append('superscript="true"')
        inner += f'<STYLE {" ".join(a)}>{_x(t)}</STYLE>' if a else _x(t)
    inner = inner.strip()
    if btype == "list_item":
        return f'<LIST type="none"><ITEM level="0">{inner}</ITEM></LIST>'
    at = []
    if btype in ("title", "section_header", "caption"):
        cx = (rect.x0 + rect.x1) / 2
        if abs(cx - pw / 2) < 0.04 * pw and rect.width < 0.8 * pw: at.append('align="center"')
    if btype == "title": at.append('heading="1"')
    if btype == "section_header": at.append('heading="2"')
    if size and abs(size - body_size) >= 1.0: at.append(f'size="{round(size * 2) / 2:g}"')
    if pb: at.append('bold="true"')
    if pi: at.append('italic="true"')
    return f'<P {" ".join(at)}>{inner}</P>' if at else f"<P>{inner}</P>"

def _table_markup(page, rect):
    try:
        tabs = page.find_tables(clip=rect).tables
        if not tabs: return None
        rows = tabs[0].extract()
        out = '<TABLE border="1">'
        for ri, r in enumerate(rows):
            out += f'<ROW{" header=" + chr(34) + "true" + chr(34) if ri == 0 else ""}>' + "".join(f"<CELL>{_x((c or '').replace(chr(10), ' ').strip())}</CELL>" for c in r) + "</ROW>"
        return out + "</TABLE>"
    except Exception:
        return None

def _label_image(pno, regs):
    img = render_page(pno).copy(); k = DPI / 72
    for r in regs:
        x0, y0, x1, y1 = [int(v * k) for v in (r["rect"].x0, r["rect"].y0, r["rect"].x1, r["rect"].y1)]
        cv2.rectangle(img, (x0, y0), (x1, y1), (30, 110, 220), 2)
        cv2.putText(img, r["id"], (x0 + 3, y0 + 22), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 220), 2, cv2.LINE_AA)
    return img

def _order_page(pno, regs):
    pw, ph = _PAGE_WH[pno]
    lst = []
    for r in regs:
        e = {"id": r["id"], "type": r["type"], "bbox": [round(r["rect"].x0 / pw, 3), round(r["rect"].y0 / ph, 3), round(r["rect"].x1 / pw, 3), round(r["rect"].y1 / ph, 3)]}
        if r["type"] not in ("table", "picture", "formula"):
            t = r["plain"]
            if len(t) <= 300: e["text"] = t
            else: e["text_start"], e["text_end"] = t[:150], t[-150:]
        lst.append(e)
    ids = [r["id"] for r in regs]
    if len(ids) < 2: return ids, "single"
    content = [{"type": "text", "text": DIGITAL_ORDER_PROMPT + "\n\nRegions:\n" + json.dumps(lst, ensure_ascii=False)},
               {"type": "image_url", "image_url": {"url": _img_uri(_label_image(pno, regs), max_side=1400)}}]
    try:
        got = [str(i) for i in (_json_from(_or_chat(M["digital_order"], content, json_mode=True, max_tokens=1500)).get("order") or [])]
        seen = set(); order = []
        for i in got:
            if i in ids and i not in seen: seen.add(i); order.append(i)
        miss = [i for i in ids if i not in seen]
        return order + miss, ("ok" if not miss else f"repaired, appended {miss}")
    except Exception as e:
        return ids, f"LLM order failed ({type(e).__name__}: {str(e)[:80]}) → geometric order"

def run_digital_english():
    set_stage("digital", "heron_layout")
    layout = heron_layout()
    set_stage("digital", "extract")
    pages = {}
    allsz = []
    for pno in page_indices:
        regs = []
        for n, it in enumerate(layout.get(pno, []), 1):
            r = {"id": f"B{n}", "type": it["type"], "rect": it["rect"]}
            if it["type"] not in _FIGURE and it["type"] != "table":
                r["segs"], r["plain"], r["size"], nch = _read_region(doc[pno], it["rect"])
                if it["type"] == "text": allsz += [r["size"]] * nch
            else:
                r["plain"] = ""
            regs.append(r)
        pages[pno] = regs
    body_size = statistics.median(allsz) if allsz else 11.0
    set_stage("digital", "order")
    with ThreadPoolExecutor(ORDER_WORKERS) as ex:
        res = list(ex.map(lambda p: _order_page(p, pages[p]), page_indices))
    blocks = []
    for pno, (order, note) in zip(page_indices, res):
        print(f"  page {pno+1}: {len(pages[pno])} blocks · order {note}")
        pw, ph = doc[pno].rect.width, doc[pno].rect.height; k = DPI / 72
        by = {r["id"]: r for r in pages[pno]}
        for rid in order:
            r = by[rid]; t = r["type"]; rect = r["rect"]
            bn = [rect.x0 / pw, rect.y0 / ph, rect.x1 / pw, rect.y1 / ph]
            bid, wh = f"p{pno+1}_{rid}", (pw * k, ph * k)
            if t in _FURNITURE:
                blocks.append(_mkblock(bid, pno + 1, bn, t, "", None, wh)); continue
            tm = _table_markup(doc[pno], rect) if t == "table" else None
            if t in _FIGURE or (t == "table" and tm is None):
                path = CROP_DIR / f"{stem}_p{pno+1}_{rid}.png"
                doc[pno].get_pixmap(clip=rect, dpi=DPI).save(str(path))
                blocks.append(_mkblock(bid, pno + 1, bn, t, "", str(path), wh)); continue
            mk = tm if tm else (_markup(t, r["segs"], r["size"], body_size, rect, pw) if r["plain"] else "")
            blocks.append(_mkblock(bid, pno + 1, bn, t, mk, None, wh))
    set_stage("-", "-")
    return blocks

if ROUTE == "digital_english":
    ordered_blocks = run_digital_english()
    print(f"digital_english: {len(ordered_blocks)} blocks")
else:
    print("skip digital_english")


Loading weights: 100%|██████████| 770/770 [00:00<00:00, 6392.63it/s]


  page 1: 14 blocks · order ok
  page 2: 15 blocks · order ok
  page 3: 12 blocks · order ok
  page 4: 12 blocks · order ok
  page 5: 19 blocks · order ok
  page 6: 12 blocks · order ok
Consider using the pymupdf_layout package for a greatly improved page layout analysis.
  page 7: 15 blocks · order ok
  page 8: 11 blocks · order ok
  page 9: 18 blocks · order ok
  page 10: 17 blocks · order ok
  page 11: 13 blocks · order ok
  page 12: 20 blocks · order ok
  page 13: 23 blocks · order ok
  page 14: 20 blocks · order ok
  page 15: 10 blocks · order ok
  page 16: 10 blocks · order ok
  page 17: 18 blocks · order ok
  page 18: 10 blocks · order ok
  page 19: 7 blocks · order ok
  page 20: 3 blocks · order ok
digital_english: 279 blocks


## ROUTE 2 — single Gemini call (digital non-English, scanned English/Hindi)

In [ ]:
# ─── ROUTE 2: single Gemini call per page (OCR + order + tags) ─────────────
# Used for: digital non-English  and  scanned English/Hindi.   [review step plugs in after this later]
SINGLE_HEAD = """You are a document-reconstruction system. You get ONE page image. In a SINGLE pass: (1) transcribe every piece of text exactly as printed, in its original script(s) — never translate, transliterate, correct spelling or invent text; (2) decide the natural reading order (columns left to right, each top to bottom; full-width blocks between; a caption right after its figure; footnotes after body); (3) tag how each block looks, for a Word-document builder.

OUTPUT: ONLY a sequence of blocks in reading order, no preamble, no code fences:
<BLOCK id="b1" kind="text|title|header|list|table|caption|footnote|picture|formula" bbox="x0,y0,x1,y1">…tagged text…</BLOCK>
- bbox is normalised 0-1 from the top-left of the page image (needed for every block; essential for picture/formula).
- picture / figure / formula / signature / stamp blocks: leave the body EMPTY (<BLOCK id=… kind="picture" bbox=…></BLOCK>) — they are cropped from the image. Do not describe them.
- Escape & < > inside text as &amp; &lt; &gt;.
- PAGE FURNITURE (scanner watermarks like "Scanned by …", bare page numbers, "P.T.O.", running headers/footers repeated on every page) goes inside <DROP>…</DROP> within its own block. Mark only what you are sure is furniture.
- If the first block starts mid-sentence or the last block is cut off, keep it exactly as is: do not complete it, do not join it to anything.
- Merge lines of one paragraph into ONE block/<P>; use <BR/> only for intentional line breaks. Keep one block per paragraph / heading / list / table / question.

TAG GRAMMAR for the block body:
"""
SINGLE_TAIL = """
Order of tags inside a block is free but every block must contain only tags + the page's own text. Never output anything outside <BLOCK>…</BLOCK>."""
SINGLE_PROMPT = SINGLE_HEAD + TAG_GRAMMAR + SINGLE_TAIL

_BLOCK_RE = re.compile(r'<BLOCK\s+id="([^"]+)"([^>]*)>(.*?)</BLOCK>', re.S)
def _attr(s, k):
    m = re.search(k + r'="([^"]*)"', s); return m.group(1) if m else None

def _single_page(pno):
    img = render_page(pno); H, W = img.shape[:2]
    txt = _or_chat(M["single_call"], [{"type": "text", "text": SINGLE_PROMPT}, {"type": "image_url", "image_url": {"url": _img_uri(img, 1800)}}],
                   timeout=300, max_tokens=16000)
    out = []
    for n, (bid, attrs, body) in enumerate(_BLOCK_RE.findall(txt), 1):
        kind = (_attr(attrs, "kind") or "text").lower()
        bb = _attr(attrs, "bbox")
        try:
            bn = [max(0.0, min(1.0, float(v))) for v in bb.split(",")][:4]
            if len(bn) != 4 or bn[2] <= bn[0] or bn[3] <= bn[1]: bn = None
        except Exception: bn = None
        b_id = f"p{pno+1}_{re.sub(r'[^A-Za-z0-9_]', '', bid) or n}"
        if kind in ("picture", "figure", "formula", "image") and bn:
            path = CROP_DIR / f"{stem}_p{pno+1}_{b_id}.png"
            x0, y0, x1, y1 = int(bn[0]*W), int(bn[1]*H), int(bn[2]*W), int(bn[3]*H)
            cv2.imwrite(str(path), img[y0:y1, x0:x1])
            out.append(_mkblock(b_id, pno + 1, bn, kind, "", str(path), (W, H)))
        else:
            out.append(_mkblock(b_id, pno + 1, bn, kind, _strip_drop(body), None, (W, H)))
    if not out:                                                    # model ignored the format → keep raw text, strip tags
        raw = re.sub(r"<[^>]+>", "", txt).strip()
        out = [_mkblock(f"p{pno+1}_raw", pno + 1, None, "text", f"<P>{_x(raw)}</P>" if raw else "", None, (W, H))]
        print(f"  page {pno+1}: no <BLOCK> output parsed → raw text kept")
    return out

def run_single_gemini():
    set_stage("single", "page_calls")
    def safe(p):
        try: return _single_page(p)
        except Exception as e:
            print(f"  page {p+1} FAILED: {type(e).__name__}: {str(e)[:150]}")
            return [_mkblock(f"p{p+1}_failed", p + 1, None, "text", "", None)]
    with ThreadPoolExecutor(ORDER_WORKERS) as ex:
        per_page = list(ex.map(safe, page_indices))
    set_stage("-", "-")
    for p, bl in zip(page_indices, per_page): print(f"  page {p+1}: {len(bl)} blocks")
    return [b for bl in per_page for b in bl]

if ROUTE == "single_gemini":
    ordered_blocks = run_single_gemini()
    print(f"single_gemini: {len(ordered_blocks)} blocks")
else:
    print("skip single_gemini")


skip single_gemini


## ROUTE 3 — scanned Indic: Bodhan OCR → dedup → LLM ordering → Qwen tags
The next two cells are your existing Stage 1 and Stage 2 code, unchanged except that they run only when `ROUTE == "scanned_indic"`.
Models come from `MODEL_OPTIONS` (`indic_order`, `indic_format`, `ocr_fallback`).

In [ ]:
if ROUTE == "scanned_indic":
    # ─── CELL 4: Stage 1 — Bodhan OCR + Python dedup + labeled image ─────────

    # ── OCR primary + fallback ───────────────────────────────────────────────
    def _b64_png(bgr):
        return base64.b64encode(cv2.imencode(".png", bgr)[1].tobytes()).decode()


    def _call_bodhan(bgr, api_key, max_tokens=8192, max_retries=3):
        payload = {
            "model": "indic-ocr",
            "messages": [{"role": "user", "content": [
                {"type": "image_url",
                 "image_url": {"url": f"data:image/png;base64,{_b64_png(bgr)}"}}
            ]}],
            "max_tokens": max_tokens,
        }
    
        for attempt in range(max_retries):
            r = requests.post(BODHAN_URL,
                              headers={"Authorization": f"Bearer {api_key}"},
                              json=payload, timeout=(10, 300))
        
            if r.status_code == 429 and attempt < max_retries - 1:
                print(f"  ⚠ Bodhan rate limit (429); waiting 5s (retry {attempt + 1}/{max_retries})...")
                time.sleep(5)
                continue
            
            r.raise_for_status()
            data = r.json()
            if "error" in data:
                raise RuntimeError(f"bodhan: {data['error'].get('message')}")
            data["_engine"] = "bodhan/indic-ocr"
            return data

    _FALLBACK_OCR_PROMPT = (
        "You are an OCR engine. Transcribe this document page faithfully.\n"
        "Return ONLY JSON (no fences, no commentary) matching this schema:\n"
        "{\n"
        '  "blocks": [\n'
        "    {\n"
        '      "order": 1,\n'
        '      "type": "Text" | "Title" | "SectionHeader" | "List" | "Caption"\n'
        '              | "Table" | "Picture" | "PageHeader" | "PageFooter" | "Footnote",\n'
        '      "bbox_norm": [x0, y0, x1, y1] in [0,1] top-left, or null if uncertain,\n'
        '      "text": "exact content, preserve Latin+Devanagari+Telugu+... scripts"\n'
        "    }\n"
        "  ]\n"
        "}\n"
        "Rules: emit in natural reading order. Picture/signature/stamp blocks get text=\"\". "
        "Render tables as GitHub-style pipe tables. Never fabricate bbox coordinates — use null.\n"
        "\n"
        "Classify these as PageFooter / PageHeader (NOT Text) — they are page furniture, "
        "not body content, and must not be joined with the body across page breaks:\n"
        "  • scanner watermarks: \"Scanned by CamScanner\", \"Scanned with OKEN Scanner\"\n"
        "  • page codes / numbers: \"1-2-103\", \"Page 5 of 10\", \"[1]\"\n"
        "  • page-turn hints: \"[P.T.O.]\", \"Please Turn Over\", \"Continued on next page\"\n"
        "  • signatures or handwritten margin marks\n"
    )

    def _call_openrouter_ocr(bgr, api_key, model, timeout=180):
        H, W = bgr.shape[:2]
        payload = {
            "model": model, "temperature": 0,
            "messages": [{"role": "user", "content": [
                {"type": "text", "text": _FALLBACK_OCR_PROMPT},
                {"type": "image_url",
                 "image_url": {"url": f"data:image/png;base64,{_b64_png(bgr)}"}},
            ]}],
        }
        r = requests.post(OPENROUTER_URL,
                          headers={"Authorization": f"Bearer {api_key}",
                                   "Content-Type": "application/json"},
                          json=payload, timeout=timeout)
        r.raise_for_status()
        data = r.json()
        if "error" in data:
            raise RuntimeError(f"openrouter: {data['error']}")
        parsed = _parse_json_object(data["choices"][0]["message"]["content"])
        blocks = []
        for i, b in enumerate(parsed.get("blocks") or [], start=1):
            bn = b.get("bbox_norm")
            bbox_px = None
            if isinstance(bn, (list, tuple)) and len(bn) == 4:
                try:
                    x0, y0, x1, y1 = [max(0.0, min(1.0, float(v))) for v in bn]
                    bbox_px = [round(x0*W, 1), round(y0*H, 1), round(x1*W, 1), round(y1*H, 1)]
                    if bbox_px[2] <= bbox_px[0] or bbox_px[3] <= bbox_px[1]: bbox_px = None
                except Exception: bbox_px = None
            blocks.append({"order": b.get("order", i), "type": b.get("type", "Text"),
                           "text": b.get("text", "") or "", "bbox_xyxy": bbox_px})
        return {"blocks": blocks, "_engine": f"openrouter/{model}"}

    def _extract_bodhan_blocks(resp):
        if isinstance(resp.get("blocks"), list): return resp["blocks"]
        content = resp.get("choices", [{}])[0].get("message", {}).get("content")
        if isinstance(content, list): return content
        if isinstance(content, str):
            parsed = _parse_json_object(content)
            if isinstance(parsed, dict) and isinstance(parsed.get("blocks"), list):
                return parsed["blocks"]
            if isinstance(parsed, list): return parsed
        if isinstance(resp.get("data"), dict) and isinstance(resp["data"].get("blocks"), list):
            return resp["data"]["blocks"]
        return []

    def ocr_with_fallback(bgr):
        try:
            resp = _call_bodhan(bgr, BODHAN_KEY)
            raw = _extract_bodhan_blocks(resp)
            normalised = []
            for i, b in enumerate(raw, start=1):
                if not isinstance(b, dict): continue
                normalised.append({"order": b.get("order", i),
                                   "type":  b.get("type", "Text"),
                                   "text":  b.get("text", "") or "",
                                   "bbox_xyxy": b.get("bbox_xyxy") or b.get("bbox")})
            normalised = [
                b for b in normalised
                if (b.get("text") or "").strip()
                or b.get("type") in _PICTURE_TYPES
            ]
            if not normalised or not any((b.get("text") or "").strip() for b in normalised):
                raise RuntimeError(f"Bodhan returned no usable blocks "
                                   f"(got {len(normalised)}; top keys={list(resp.keys())[:6]})")
            return {"blocks": normalised, "_engine": "bodhan/indic-ocr"}
        except Exception as e:
            print(f"  Bodhan failed: {type(e).__name__}: {str(e)[:160]} — trying Gemini")
            if not OPENROUTER_KEY: raise RuntimeError("Bodhan failed and OPENROUTER_API_KEY is not set.")
            return _call_openrouter_ocr(bgr, OPENROUTER_KEY, OCR_FALLBACK_MODEL)


    # ── Python-side geometric dedup (unchanged from the previous notebook) ────
    _PICTURE_TYPES = {"Picture", "Chart", "Diagram", "Image", "Figure", "picture"}

    def _area(b): return max(0, b[2] - b[0]) * max(0, b[3] - b[1])
    def _inter(a, b):
        x0, y0 = max(a[0], b[0]), max(a[1], b[1])
        x1, y1 = min(a[2], b[2]), min(a[3], b[3])
        return max(0, x1 - x0) * max(0, y1 - y0)
    def _is_informative(block):
        if (block.get("text") or "").strip(): return True
        return block.get("type") in _PICTURE_TYPES
    def _normalize_words(text):
        tbl = str.maketrans('', '', string.punctuation + "\u0964\u0965")
        return {w.translate(tbl).lower() for w in (text or "").split() if w.strip()}

    def dedup_boxes(blocks, iou_threshold=0.3, contain_threshold=0.95, text_subsume_threshold=0.6):
        sorted_blocks = sorted(blocks,
            key=lambda b: (_is_informative(b), _area(b.get("bbox_xyxy") or [0,0,0,0])),
            reverse=True)
        kept, removed = [], []
        for c in sorted_blocks:
            if not c.get("bbox_xyxy"): kept.append(c); continue
            a = _area(c["bbox_xyxy"]) or 1.0
            c_words = _normalize_words(c.get("text") or "")
            drop_reason = None
            for k in kept:
                if not k.get("bbox_xyxy"): continue
                inter = _inter(c["bbox_xyxy"], k["bbox_xyxy"])
                if inter / a >= contain_threshold:
                    drop_reason = f"contained in B{k.get('order')} ({inter/a:.0%})"; break
                ka = _area(k["bbox_xyxy"]) or 1.0
                iou = inter / (a + ka - inter)
                if iou >= iou_threshold:
                    drop_reason = f"IoU {iou:.2f} with B{k.get('order')}"; break
                if inter > 0 and c_words:
                    k_words = _normalize_words(k.get("text") or "")
                    if k_words:
                        ov = len(c_words & k_words) / len(c_words)
                        if ov >= text_subsume_threshold:
                            drop_reason = f"text {ov:.0%} subsumed by B{k.get('order')}"
            if drop_reason: c = {**c, "_drop_reason": drop_reason}; removed.append(c)
            else: kept.append(c)
        kept.sort(key=lambda b: b.get("order", 0))
        return kept, removed


    # ── draw B0, B1, ... labels on the preprocessed image ────────────────────
    _PALETTE = [(230, 25, 75), (60, 180, 75), (0, 130, 200), (245, 130, 48),
                (145, 30, 180), (0, 128, 128), (240, 50, 230), (170, 110, 40)]

    def draw_labeled(img_bgr, blocks):
        pil = Image.fromarray(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
        draw = ImageDraw.Draw(pil)
        try: font = ImageFont.truetype("arial.ttf", 24)
        except Exception: font = ImageFont.load_default()
        types = sorted({b.get("type", "?") for b in blocks})
        color_for = {t: _PALETTE[i % len(_PALETTE)] for i, t in enumerate(types)}
        W, H = pil.size
        for b in blocks:
            if not b.get("bbox_xyxy"): continue
            x0, y0, x1, y1 = [int(v) for v in b["bbox_xyxy"]]
            c = color_for[b.get("type", "?")]
            draw.rectangle([x0, y0, x1, y1], outline=c, width=3)
            tag = b["_label"]
            tw  = int(draw.textlength(tag, font=font)); th, pad = 28, 4
            tag_x = x0 - pad; tag_y = y0 - th - 2
            if tag_y < 0: tag_y = 0
            if tag_x < 0: tag_x = 0
            if tag_x + tw + 2*pad > W: tag_x = W - tw - 2*pad
            draw.rectangle([tag_x, tag_y, tag_x + tw + 2*pad, tag_y + th], fill=c)
            draw.text((tag_x + pad, tag_y + 2), tag, fill="white", font=font)
        return pil


    # ── Bodhan type → standard region type (used by the sidecar + builder) ──
    BODHAN_TO_STD = {
        "Title":"title", "SectionHeader":"heading", "Text":"paragraph",
        "List":"list_item", "Caption":"caption", "Table":"Table",
        "Picture":"picture", "Chart":"picture", "Diagram":"picture",
        "Image":"picture", "Figure":"picture",
        "PageHeader":"header", "PageFooter":"footer", "PageNumber":"footer",
        "Header":"header", "Footer":"footer", "Footnote":"paragraph",
        "Infobox":"paragraph", "Folio":"footer",
    }


    # ─────────────────────────────────────────────────────────────────────────
    # Run Stage 1: OCR + dedup + label + build page_dicts (RAW order from Bodhan)
    # ─────────────────────────────────────────────────────────────────────────
    reset_metrics()
    ocr_responses, preprocessed = {}, {}
    ocr_failures = {}

    # page_dicts holds blocks in the order Bodhan emitted them (after Python dedup).
    # Stage 2 will reorder them. Each region carries a stable block_id used end-to-end.
    page_dicts = []

    def _stage1_page(pno):
        print(f"\n── Stage 1 · page {pno+1}/{total} ──")
        set_stage("A", "preprocess")
        img = preprocess_page(pno)
        preprocessed[pno] = img

        set_stage("A", "ocr")
        try:
            resp = ocr_with_fallback(img)
            ocr_responses[pno] = resp
            print(f"  ocr ok: {resp['_engine']}  ({len(resp['blocks'])} blocks)")
            for _bi, _b in enumerate(resp["blocks"]):
                _show_block("[OCR]", _b, label=f"#{_bi}")
        except Exception as e:
            ocr_failures[pno] = str(e)
            print(f"  OCR failed: {e}")
            return None

        set_stage("A", "dedup")
        kept, removed = dedup_boxes(resp["blocks"])
        for i, b in enumerate(kept): b["_label"] = f"B{i}"
        print(f"  dedup: {len(resp['blocks'])} → kept {len(kept)}, removed {len(removed)}")
        print(f"  ── DEDUP kept (page {pno+1}) ──")
        for _b in kept: _show_block("[KEPT]", _b)
        if removed:
            print(f"  ── DEDUP removed (page {pno+1}) ──")
            for _b in removed:
                reason = _b.get("_drop_reason", "?")
                print(f"    [DEDUP-DROP] #{_b.get('order','?')} reason={reason}  text={_preview(_b.get('text'))!r}")

        set_stage("A", "label")
        labeled = draw_labeled(img, kept)
        labeled.save(PAGE_DIR / f"{stem}_p{pno+1}_labeled.png")

        # Build regions in Bodhan's natural (post-dedup) order — Stage 2 reorders.
        pil = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        H, W = img.shape[:2]
        regions = []
        for i, b in enumerate(kept, start=1):
            rid   = b["_label"]
            std   = BODHAN_TO_STD.get(b.get("type"), "paragraph")
            block_id = f"p{pno+1}_{rid}"
            reg = {
                "id":          f"A_p{pno+1}_r{i}",
                "block_id":    block_id,
                "page":        pno + 1,
                "type":        std,
                "bodhan_type": b.get("type", "Text"),
                "bbox_xyxy":   b.get("bbox_xyxy"),
                "page_wh":     [W, H],
            }
            if std == "picture" and b.get("bbox_xyxy"):
                x0, y0, x1, y1 = [int(v) for v in b["bbox_xyxy"]]
                crop_path = CROP_DIR / f"{stem}_p{pno+1}_{rid}.png"
                crop_path.parent.mkdir(parents=True, exist_ok=True)
                pil.crop((x0, y0, x1, y1)).save(crop_path)
                reg["image_path"] = str(crop_path)
                reg["text"] = ""
            else:
                reg["text"] = b.get("text") or ""
            regions.append(reg)
        return {"page": pno + 1, "regions": regions}

    with ThreadPoolExecutor(BODHAN_WORKERS) as _ex:
        page_dicts = [d for d in _ex.map(_stage1_page, page_indices) if d]   # page order kept

    set_stage("-", "-")

    # Compact summary of what will enter Stage 2.
    print(f"\n── Stage 1 summary ──")
    total_regions = sum(len(p["regions"]) for p in page_dicts)
    print(f"pages processed: {len(page_dicts)} · regions: {total_regions}")
    for pd in page_dicts:
        print(f"  page {pd['page']}: {len(pd['regions'])} region(s)")


In [ ]:
if ROUTE == "scanned_indic":
    # ─── CELL 5: Stage 2 — one combined call per chunk ────────────────────────

    import json
    (METHOD_DIR / f"{stem}__combined_cache.json").unlink(missing_ok=True)   # fresh run each time

    # ── knobs ──────────────────────────────────────────────────────────────
    HTTP_ATTEMPTS             = 5       # per API call, exponential backoff on network/429/5xx
    MAX_SPLIT_DEPTH           = 3       # on chunk reject: halve, halve, single, single@temp0.3, RAW
    NET_FAIL_BREAKER          = 3       # stop after N consecutive network-failed chunks
    ABORT_IF_NO_DNS           = True
    SHOW_REJECT_DEBUG         = True
    STRIP_LITERAL_BACKSLASH_N = True
    INSERT_PAGE_BREAKS        = False
    USE_CACHE                 = True

    # Fidelity (chunk-level, multiset / character-count because output is reordered).
    # Drop budget is intentionally larger here than in the previous notebook — the
    # whole point of this stage is to allow controlled word/character-level dedup.
    FID_DROP_TOL       = 0.10    # up to 10% of chunk chars may be dropped ...
    FID_DROP_MIN_CHARS = 15      # ... but always allow this many
    FID_ADD_TOL        = 0.05    # up to 5% may be added (bumped from 3% so one
                                 # mis-cased letter or tag-attribute spelling
                                 # doesn't cause 4 cascading retries)
    FID_ADD_MIN_CHARS  = 25      # ... with a floor of 25 chars so very small
                                 # chunks (short page headers etc.) don't trip
                                 # on a single stray character
    FID_IGNORE_CHARS   = "_$"    # ignored in comparison: blank runs, LaTeX $ delimiters


    # ═════════════════════════════════════════════════════════════════════════
    # THE PROMPT — generalisable; no domain examples, no exam/physics content
    # ═════════════════════════════════════════════════════════════════════════
    COMBINED_PROMPT = r"""
You are a document-reconstruction system. Your input is a chunk of a scanned document after an OCR step, together with the image(s) of the page(s) the chunk came from. In ONE pass you must do THREE jobs, in this order:

(A) WITHIN-BLOCK CLEANUP — verified against the image.
(B) ORDER — already decided and verified upstream. Emit the blocks in EXACTLY the order given.
(C) FORMATTING TAGS — tags that describe how each block looks on the page.

A later program consumes your tagged output and builds a Word document. Never say anything outside the <BLOCK>…</BLOCK> wrappers: no preamble, no prose, no code fences.


================================================================
INPUT
================================================================
Each block arrives as

  <BLOCK id="…" type="…" page="N" bbox="x0,y0,x1,y1">
  …OCR text of this block…
  </BLOCK>

- id    is a stable identifier; preserve it exactly in your output.
- type  is a hint from an upstream layout model and is often wrong; trust what you SEE in the image when they disagree.
- page  is the 1-indexed page the block was detected on.
- bbox  is [x0, y0, x1, y1] normalised to [0,1] over the page image. Use it to locate each block's region.
- The text is the OCR of this block. The characters & < > inside text appear as &amp; &lt; &gt;; keep them that way in your output.

Blocks arrive in their FINAL reading order. That order was decided upstream from the layout and checked against the page image. Emit the <BLOCK> wrappers in exactly the order you received them. Never reorder, never move a block above or below its neighbours, never remove a block. Ordering is NOT your job.

Along with the blocks you receive the image of pages the chunk spans. Use the image for verification and layout only — do NOT read any text from the image into your output.


================================================================
(A) WITHIN-BLOCK CLEANUP
================================================================
DEFAULT: keep every character of every block exactly as it is in the OCR.

You MAY remove characters or words from inside a block ONLY when ALL of the following hold:

  (a) The removed text is REDUNDANT in one of these two patterns, and only these two:
        - echo / stutter: the same short run appears twice back-to-back inside the block (e.g. a word, number or short phrase repeated with no intervening content);
        - overlap duplicate: another block in the SAME chunk has an overlapping bbox and already contains the same words at that visual position, so one of the two is a fragment duplicated by the upstream layout model.

  (b) The IMAGE, inspected at this block's bbox, visibly shows the surviving text ONCE and does not show the removed copy as a separate second occurrence. Count the occurrences in the image region before removing anything.

  (c) After removal, every meaningful word that the image shows for this block is still present. You never shorten or summarise content that the image genuinely contains.

If any of (a)(b)(c) is uncertain, KEEP every character. When in doubt — KEEP. Deletion is not easily allowed until it verifies with surrounding blocks and the image.

A word is not redundant just because it is short, sits on its own line, or doesn't fit the pattern of its neighbours. The overlap-duplicate rule requires the same word to appear VISIBLY TWICE in the image at bbox-overlapping regions. If you cannot see the word twice, keep it, even if it looks orphaned.

You MAY use a block's neighbours in the same chunk as context for your dedup decisions. Before deciding whether a word in block X is a redundant echo of block Y, read every block in this chunk. That is why the chunk is given to you together.

WHOLE-BLOCK EMPTYING (returning <BLOCK id="…"></BLOCK> with no content) is extremely rare and allowed ONLY when the entire block's words are a verified bbox-overlapping duplicate of another block in the SAME chunk, confirmed from the image. 

You MAY NEVER:
  - Rewrite, paraphrase, correct spelling, translate, transliterate, or complete truncated text.
  - Add any word, number, label, caption, title, heading or row that is not in the OCR of that block.
  - Transcribe text from the image into an empty block. If a block's OCR text is empty, your output for it is <BLOCK id="…"></BLOCK> — even when the image at that bbox shows real content. The image is for formatting and ordering decisions only; words in the OCR are the only source of content, in every block.
  - Move text from one block to another. Each block's output may contain only words from that block.
  - Change a block's id, add a new id, or omit a block (except for the rare verified emptying above).

Never remove text just because it is repeated. Always verify suspected duplicates against the page image and their bounding boxes first. Remove a repeated occurrence only when there is sufficient evidence that it is a duplicate — for example, when the bounding boxes overlap or are identical and the text represents the same location in the image. If the bounding boxes are identical or cannot reliably distinguish the occurrences, also consider the distance between the current and previous occurrences. Preserve legitimate repetitions, even when the same text appears multiple times in the document.

What you CAN add freely is TAGS. Tags are not content. Alignment, size, bold / italic / underline, heading levels, lists, options, tables, blanks, line breaks — all of these are instructions to a Word-document builder about how the OCR text should look. Add every tag the image evidence supports; don't add any character of text that isn't already in the block.


================================================================
(B) ORDER + PAGE FURNITURE
================================================================
ORDER: blocks arrive already in final reading order. Emit exactly one wrapper per input block, in exactly that order. Never omit a wrapper.

PAGE FURNITURE: scanner watermarks ("Scanned by …", "Scanned with …"), bare page numbers, page-turn hints ("[P.T.O.]", "Please Turn Over", "Continued on next page"), and a header/footer repeated identically on every page are NOT body content. Decide this from the block's TEXT and from the IMAGE at its bbox — the layout `type` label is only a weak hint and is often wrong. A block may be entirely furniture, entirely body, or MIXED: body text with a furniture fragment glued to it (at the start, end, or middle).
Wrap each furniture fragment in <DROP>…</DROP>, inside its own block, leaving the body text outside it exactly as in the OCR. Syntax examples:
  mixed block   : <P>real body text …</P><DROP>Scanned by …</DROP>
  all furniture : <DROP>754</DROP>
The wrapper is still emitted; only the fragment is marked. The downstream program deletes everything inside <DROP> and keeps a log, so mark only text you are sure is furniture. Being short, orphaned, a number, or on its own line is NOT enough — a number that is part of the content (a list label, a year, a marks/hours value) is body. If unsure, do NOT mark it.

Picture / Figure / Chart / Diagram blocks have no OCR text; emit <BLOCK id="…"></BLOCK> for them.


================================================================
(C) FORMATTING TAGS
================================================================
Return each block as <BLOCK id="…"> …tags… </BLOCK> (copy the id exactly; type and page are no longer needed on the wrapper). Exactly one wrapper per input block. No merging, splitting, or moving text between blocks. Output ONLY the wrappers: no preamble, no comments, no code fences.

Units: font size in points; indents in inches; spacing in points. Leave an attribute out when it is the default or when you cannot judge it.

----------------------------------------------------------------
<P …>text</P>
----------------------------------------------------------------
Every paragraph is a P. Attributes:
  align         left | center | right | justify   (default left). ALWAYS set align when visibly centered, right-aligned or justified.
  size          font size in points. Ordinary body text ≈ 11. Estimate by comparing letter height with ordinary body text on the SAME page (main title ≈ 16–22, section header ≈ 13–15, body ≈ 11, small print / notes ≈ 8–10). Leave out for body size.
  bold | italic | underline   "true" only when the WHOLE paragraph is visibly so.
  font          only if clearly identifiable (e.g. "Times New Roman", "Arial"); never for Indic scripts.
  color         "#RRGGBB" only for clearly coloured text.
  heading       1 | 2 | 3 for the document / institution title, section headers, and small sub-labels: a line that is visibly larger, bolder or centered and stands apart from body text.
  indent_left, indent_first (negative = hanging), space_before, space_after, line_spacing (1.0 = single): only when clearly different from normal.

----------------------------------------------------------------
<STYLE bold= italic= underline= strike= size= font= color="#RRGGBB" highlight= superscript= subscript=>text</STYLE>
----------------------------------------------------------------
Inline character formatting for PART of a paragraph. Use the smallest span.

----------------------------------------------------------------
<BR/>
----------------------------------------------------------------
Intentional line break inside a paragraph.

----------------------------------------------------------------
<BLANK type="underscore|dots|line|space" length="N"/>
----------------------------------------------------------------
A fill-in-the-blank run, at exactly the same spot. length = the number of characters in the OCR run (count them). Use ONLY where the OCR contains such a run (3+ underscores / dots / dashes). Never add a blank the OCR text does not have; never fill a blank with an answer.

----------------------------------------------------------------
<LIST type="bullet|number|alpha|roman|none"><ITEM level="0">…</ITEM></LIST>
----------------------------------------------------------------
Use <LIST> only when the OCR text shows a vertical column of items with a leading marker on each item.

CRITICAL — label preservation: inside each <ITEM>, the first characters of the OCR line are the item's label, and they are CONTENT, not formatting. You MUST copy them verbatim into the <ITEM> text, together with the punctuation and the single space that follows. The downstream builder does NOT auto-number list items; whatever label is missing from your <ITEM> text is missing from the final document.

What counts as a label (script-independent): any short run of 1–3 characters at the start of the OCR line, followed by a punctuation mark (one of ".", ")", "(", "-", ":", "/", "]", "}") and whitespace, AND the same shape repeats on the next sibling items in the list. Script does not matter — the label can be a digit, a Latin letter, a roman numeral, a single letter or ligature of any Indic / CJK / Cyrillic / Arabic script, a circled number, or a pure bullet ("•", "·", "-", "*", "–", "—", "◦", "▪", "➤"). Keep the entire label — its character(s), its punctuation and the space after it — verbatim in the <ITEM> text.

If a line in the OCR has no leading marker and you cannot be certain the group is a list, emit <P> for each line instead of <LIST>. Faithful content always beats a prettier layout.

----------------------------------------------------------------
<QUESTION> … </QUESTION>   optional wrapper around a numbered item; a P for the stem, then
<OPTIONS layout="vertical|grid" columns="2"><OPTION>(a) …</OPTION> …</OPTIONS>
----------------------------------------------------------------
Same label-preservation rule as <LIST>: the option marker is CONTENT, whatever script it is in. Copy it verbatim into the <OPTION> text with its following punctuation and space. Keep every option in the original order. vertical = one option per line. grid = options arranged in N columns on the page.

----------------------------------------------------------------
<TABLE border="1" align="left|center|right"><ROW header="true"><CELL align= bold= size= colspan="2">text</CELL></ROW></TABLE>
----------------------------------------------------------------
CRITICAL TABLE RULE: You MUST convert all tabular data (whether in Markdown format with `|` pipes or HTML markup) into this exact XML structure. 
- NEVER output raw Markdown pipe (`|`) characters in the final text.
- Always include border="1" so the downstream document builder applies proper grid lining.
- Keep cell text exactly as in the OCR. Ensure every row covers the same number of columns by using the colspan attribute where necessary.

----------------------------------------------------------------
Symbols
----------------------------------------------------------------
Keep all symbols, maths and Unicode exactly. Use <STYLE superscript=…> and <STYLE subscript=…> for raised / lowered characters. Never replace a formula by words.

----------------------------------------------------------------
Decision rules
----------------------------------------------------------------
  - Compare each block's text with its region of the image. Ordinary body text stays a plain <P> with no attributes.
  - A short or capitalised line is not automatically a heading; a question-style number does not make a line bold. Treat similar elements consistently within the chunk.
  - If a sentence appears to continue from one block or page into the next, leave it as is: do not join blocks and do not complete the sentence.
  - If you cannot judge a property, leave it out. Faithful content and faithful order always beat a prettier layout.


================================================================
BEFORE YOU EMIT — SELF-CHECK
================================================================
Do these four checks silently; emit the output only after all four pass.

  (1) Does every input id appear exactly once in your output? (Rare verified empties under rule (A) are still present as empty wrappers.)
  
  (2) Is the output order IDENTICAL to the input order? If not, restore the input order.

  (3) Did any deletion you performed meet rules A(a), A(b), A(c)? If any was uncertain w.r.to image, roll it back and keep the text.

  (4) Did you add any word, number or label that is not in the OCR of that block? If yes, remove it, i.e No additions.

  (5) For every <ITEM> and <OPTION> you emitted: does its text begin with the exact leading label (marker + punctuation + space) that the OCR line had, in whatever script? If the <ITEM> or <OPTION> text starts a word later than the OCR line did, the label was stripped — put it back.

  (6) You don't have to deal with spelling mistakes, just use ocr text as it is. do not correct spelling mistakes.
  
================================================================
EXAMPLE (dummy content — shows the output SYNTAX only, not domain content)
================================================================
Input (detection order; three dummy blocks):

<BLOCK id="x2" type="Text" page="1" bbox="0.08,0.42,0.92,0.52">option alpha  option beta  option gamma  option delta</BLOCK>
<BLOCK id="x0" type="Text" page="1" bbox="0.08,0.04,0.92,0.11">LOREM IPSUM CENTRE</BLOCK>
<BLOCK id="x1" type="SectionHeader" page="1" bbox="0.08,0.30,0.92,0.36">Section One</BLOCK>

Output (reading order; label before its content; nothing invented):

<BLOCK id="x0">
<P align="center" heading="1" size="18" bold="true">LOREM IPSUM CENTRE</P>
</BLOCK>
<BLOCK id="x1">
<P align="left" heading="2" size="13" bold="true">Section One</P>
</BLOCK>
<BLOCK id="x2">
<OPTIONS layout="grid" columns="2">
<OPTION>option alpha</OPTION>
<OPTION>option beta</OPTION>
<OPTION>option gamma</OPTION>
<OPTION>option delta</OPTION>
</OPTIONS>
</BLOCK>


Now process the chunk below.

INPUT:"""

    _PROMPT_HASH = hashlib.sha1(COMBINED_PROMPT.encode("utf-8")).hexdigest()[:12]


    # ═════════════════════════════════════════════════════════════════════════
    # PAGE ORDER — one compact call per page + downstream validation
    # ═════════════════════════════════════════════════════════════════════════
    ORDER_MODEL        = (lambda m: m if isinstance(m, str) else m[0])(globals().get("order_model", COMBINED_MODELS))
    ORDER_BAD_FRAC     = 0.20   # reject the answer if >20% of ids had to be repaired
    ORDER_TAIL_IGNORE  = 3      # upward jumps among the last N positions are expected (furniture goes last)

    PLAN_PROMPT = """
You are a deterministic, language-agnostic document layout and reading-order parser.
You are given:
1. A scanned page image with labeled bounding boxes (matching the block IDs).
2. OCR block metadata lines under `Regions:`:
   `id | x0,y0,x1,y1 (% of page, top-left origin) | type | text snippet`
   (Note: input detection order is NOT the reading order).

Execute this 4-step algorithm in exact sequence:

======================================================================
STEP 1: DETECT LAYOUT & GROUP ATOMIC PAIRS
======================================================================
1. Check horizontal X-coordinates (`x0, x1` in %) of body text blocks:
   - "2col": Two parallel vertical columns of body text sit side-by-side (Left column `x1 <= 55`, Right column `x0 >= 45`).
   - "1col": Single vertical flow of full-width text, centered headers, or lists.
   - TRAP WARNING: If a page has full-width paragraphs (`x0 < 20` and `x1 > 80`) and only wraps around an inset side Figure+Caption on part of the page, classify as "1col" (NOT "2col").
2. Group Atomic Pairs & Horizontal Rows:
   - Pair every `[picture]` / Figure / Table with its adjacent Caption (`[Figure, Caption]` or `[Table_Caption, Table]`). Never separate them.
   - HORIZONTAL BANDS (Forms/Exams/Syllabi): If multiple small blocks sit on roughly the same horizontal row (`|y0_A - y0_B| <= 3`), order them strictly Left-to-Right (`x0` ascending) before moving down.
   - OVERLAPPING/SPLIT HEADINGS: If a small heading/label box overlaps or sits at the immediate top-left of a content box (e.g., a unit label split from its text), always place the heading box immediately BEFORE the content box.

======================================================================
STEP 2: CHECK TOP-OF-PAGE CONTINUATION (FLOAT BYPASS RULE)
======================================================================
1. Identify `FIRST_BODY_BLOCK`:
   - In "1col": The topmost body `Text` block (skipping top Figures, Pictures, Tables, Captions, and Page Headers).
   - In "2col": The topmost body `Text` block in the LEFT column (skipping top full-width Figures, Captions, Title, and Author blocks).
2. Inspect the very start of `FIRST_BODY_BLOCK`'s text snippet:
   - Bicameral scripts (English/Latin): Starts with a lowercase letter (`a-z`) or hyphenated word fragment.
   - Unicameral/Indic scripts (Tamil, Telugu, etc.): Starts mid-sentence, or starts with a continued numbered/lettered list item (e.g., starts at `4.` when `1-3` were on the previous page, or finishes a previous syllabus unit above a new main title).
3. IF `FIRST_BODY_BLOCK` is a continuation from the previous page AND there is a floating `[Figure, Caption]` or `[Table_Caption, Table]` physically above it (`y0_float < y0_FIRST_BODY_BLOCK`):
   - ACTION: Move `FIRST_BODY_BLOCK` to the VERY FRONT of the reading order (index 0), jumping ahead of the top float(s). Immediately after `FIRST_BODY_BLOCK`, emit the bypassed top float(s) in their original order, and then resume the rest of the column/page.
   - ELSE: Keep standard top-to-bottom zone order.

======================================================================
STEP 3: RESOLVE 2-COLUMN FLOW, WRAPPED FIGURES & CUT-OFFS
======================================================================
1. IF `layout_type == "2col"`:
   - Read full-width top headers/floats first (unless bypassed in Step 2).
   - Read the entire LEFT column top-to-bottom (`y0` ascending).
   - Then read the entire RIGHT column top-to-bottom (`y0` ascending).
   - TRAP WARNING: If the bottom block of the LEFT column ends mid-sentence/hyphenated AND the top block of the RIGHT column starts with a lowercase word, that is a normal intra-page transition (`Left_Bottom -> Right_Top`). NEVER move `Right_Top` to the start of the page.
   - FOOTNOTE / ABSTRACT RULE: If a column has a footnote at the bottom, but the last body paragraph of that column is cut off mid-sentence, place the footnote BEFORE the cut-off paragraph (or after the completion of an Abstract split).
2. IF `layout_type == "1col"` with an Inline Wrapped Figure:
   - Never insert a `[Figure, Caption]` pair inside an unfinished sentence. Place the `[Figure, Caption]` pair immediately after the complete paragraph that references it (or between two complete paragraphs).
3. Footer & Margin Anchors:
   - Place bare page numbers, `[P.T.O.]`, and scanner watermarks (`Scanned by CamScanner`) at the very end of the reading order (unless the final body block is cut off mid-sentence at page end, in which case furniture goes just before the cut-off block).

======================================================================
STEP 4: OUTPUT FORMAT (STRICT JSON ONLY — NO MARKDOWN FENCES)
======================================================================
Every input `id` MUST appear in `"order"` exactly once using the exact string ID given (e.g., `"p1_B0"`). Fill the JSON keys in this exact order:
{
  "layout_type": "1col or 2col",
  "step1_zones_and_pairs": "List top floats/headers, atomic [Figure, Caption] pairs, horizontal bands, or overlapping split boxes.",
  "step2_first_body_check": "State the ID of FIRST_BODY_BLOCK, quote its first 3 words, state if it continues from the previous page, and state if it jumps ahead of any top floats.",
  "step3_column_and_cutoff_check": "For 2col: check Left-Bottom to Right-Top transition. For 1col: check wrapped figures or split OCR lines. List any blocks ending mid-sentence.",
  "first_text_block_starts_mid_flow": true,
  "starts_mid": ["ids of blocks starting mid-sentence or mid-list"],
  "cut_off": ["ids of blocks ending mid-sentence or split across boxes"],
  "moves": "1-sentence summary of reorders made",
  "order": ["id", "id", "..."]
}
"""

    BBOX_NOTE = (
        "\nEach line under Regions has format: `id | x0,y0,x1,y1 | type | text snippet` "
        "where `x0,y0,x1,y1` are percentages (0 to 100) from the top-left corner.\n"
    )

    REGIONS_HEADER = "\nRegions:\n"


    def _pct(b):
        bb, wh = b.get("bbox_xyxy"), b.get("page_wh")
        if not bb or not wh: return "?"
        return ",".join(str(round(100 * v / d)) for v, d in zip(bb, (wh[0], wh[1], wh[0], wh[1])))

    def _brief(b, n=50):
        t = " ".join((b.get("text") or "").split())
        if b.get("is_picture"): return "[picture]"
        return t if len(t) <= 2 * n + 3 else f"{t[:n]} … {t[-n:]}"

    def _labeled_page_image_data_uri(page_1based, blocks, max_side=1024, jpeg_quality=85):
        img = render_page(page_1based - 1)
        if img is None:
            raise RuntimeError(f"render_page({page_1based - 1}) returned None")
        img = img.copy()
        h, w = img.shape[:2]
        for b in blocks:
            bb = b.get("bbox_xyxy")
            if not bb: continue
            x0, y0, x1, y1 = [int(round(v)) for v in bb]
            btype = (b.get("bodhan_type") or "Text").lower()
            if b.get("is_picture") or "pic" in btype or "fig" in btype:
                color = (40, 160, 40)
            elif "tab" in btype or "head" in btype or "title" in btype:
                color = (200, 90, 20)
            elif "cap" in btype:
                color = (160, 30, 120)
            else:
                color = (30, 110, 220)
            cv2.rectangle(img, (x0, y0), (x1, y1), color, 2)
            short_id = b["block_id"].split("_", 1)[-1]
            (tw, th), _ = cv2.getTextSize(short_id, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2)
            ly0 = max(0, y0 - th - 6)
            cv2.rectangle(img, (x0, ly0), (x0 + tw + 8, ly0 + th + 6), color, -1)
            cv2.putText(img, short_id, (x0 + 4, ly0 + th + 1),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2, cv2.LINE_AA)

        scale = min(1.0, max_side / max(h, w))
        if scale < 1.0:
            img = cv2.resize(img, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_AREA)
        ok, buf = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, jpeg_quality])
        if not ok:
            raise RuntimeError(f"cv2.imencode failed for labeled page {page_1based}")
        return "data:image/jpeg;base64," + base64.b64encode(buf.tobytes()).decode("ascii")

    def _repair_order(got, want):
        """Make the model's list a valid permutation. Returns (order, n_unknown_or_dup, missing)."""
        short_to_full = {w.split("_", 1)[-1]: w for w in want}
        ws, seen, clean = set(want), set(), []
        for raw_i in got:
            i = raw_i if raw_i in ws else short_to_full.get(raw_i, raw_i)
            if i in ws and i not in seen:
                clean.append(i); seen.add(i)
        n_bad   = len(got) - len(clean)
        missing = [i for i in want if i not in seen]
        for m in missing:                       # put it after its nearest detect-order predecessor
            k, pos = want.index(m), 0
            for j in range(k - 1, -1, -1):
                if want[j] in clean: pos = clean.index(want[j]) + 1; break
            clean.insert(pos, m)
        return clean, n_bad, missing

    def _h_frac(a, b):
        ov = min(a[2], b[2]) - max(a[0], b[0])
        return 0.0 if ov <= 0 else ov / max(1.0, min(a[2] - a[0], b[2] - b[0]))

    def _order_flags(ordered, starts_mid_ids=()):
        """Report consecutive blocks in one column where the next block lies entirely ABOVE the previous,
    IGNORING intentional Step 2 float-bypass moves (where `a` is in starts_mid_ids)."""
        starts_mid_set = set(starts_mid_ids or ())
        flags = []
        for k, (a, b) in enumerate(zip(ordered, ordered[1:])):
            if a["block_id"] in starts_mid_set or a["block_id"].split("_", 1)[-1] in starts_mid_set:
                continue
            A, B = a.get("bbox_xyxy"), b.get("bbox_xyxy")
            if A and B and _h_frac(A, B) >= 0.5 and B[3] < A[1] and k + 1 < len(ordered) - ORDER_TAIL_IGNORE:
                flags.append((a["block_id"], b["block_id"]))
        return flags

    def _ask_order(messages):
        body = {
            "model": ORDER_MODEL,
            "temperature": 0.0,
            "response_format": {"type": "json_object"},
            "messages": messages,
        }
        data = _post_with_retry(body, 240)
        if "error" in data: raise RuntimeError(f"openrouter: {data['error']}")
        txt = data["choices"][0]["message"]["content"] or ""
        parsed_json = _parse_json_object(txt)
        order_list = [str(x) for x in (parsed_json.get("order") or [])]
        starts_mid = [str(x) for x in (parsed_json.get("starts_mid") or [])]
        return txt, order_list, starts_mid

    def _validated(got, blocks):
        want = [b["block_id"] for b in blocks]
        order, n_bad, missing = _repair_order(got, want)
        if n_bad + len(missing) > max(2, ORDER_BAD_FRAC * len(want)):
            raise ValueError(f"too broken to repair (unknown/dup={n_bad}, missing={missing})")
        by_id = {b["block_id"]: b for b in blocks}
        return [by_id[i] for i in order], n_bad, missing

    def _llm_page_order(page_no, blocks, include_bbox=True):
        lines = "\n".join(
            f'{b["block_id"]} | {_pct(b)} | {b.get("bodhan_type", "Text")} | {_brief(b)}'
            for b in blocks
        )
        full_prompt = (
            PLAN_PROMPT
            + (BBOX_NOTE if include_bbox else "")
            + REGIONS_HEADER
            + lines
        )
        labeled_uri = _labeled_page_image_data_uri(page_no, blocks)
        first = [{"role": "user", "content": [
                    {"type": "text", "text": full_prompt},
                    {"type": "image_url", "image_url": {"url": labeled_uri}}]}]
        txt, got, starts_mid = _ask_order(first)
        final, n_bad, missing = _validated(got, blocks)
        notes = []
        if n_bad or missing: notes.append(f"repaired (unknown/dup={n_bad}, missing={missing})")

        flags = _order_flags(final, starts_mid_ids=starts_mid)
        if flags:
            pairs = "; ".join(f"{a} then {b}" for a, b in flags)
            ask = ("Check your order. In these consecutive pairs the second block lies entirely above the first, inside the same column: "
                   f"{pairs}. For each pair, keep it only if intended (continuation jump, caption/figure pairing, overlapping split header, or furniture last); "
                   "otherwise fix it. Reply with the full JSON object again.")
            try:
                _, got2, _ = _ask_order(first + [{"role": "assistant", "content": txt}, {"role": "user", "content": ask}])
                final2, b2, m2 = _validated(got2, blocks)
                left = _order_flags(final2, starts_mid_ids=starts_mid)
                notes.append(f"flags {flags} → re-check " + ("changed order" if final2 != final else "kept order") +
                             (f", still flagged {left}" if left else ""))
                final = final2
            except Exception as e:
                notes.append(f"flags {flags}; re-check failed ({type(e).__name__}) → kept first answer")
        return final, "; ".join(notes) or "ok"

    def _order_all_pages(sendable):
        by_page = defaultdict(list)
        for b in sendable: by_page[b["page"]].append(b)          # detect order preserved
        short = lambda L: [b["block_id"].split("_", 1)[1] for b in L]
        def _one(pno):
            det = by_page[pno]; final, note = det, "single block"
            if len(det) > 1:
                try: final, note = _llm_page_order(pno, det)
                except Exception as e:
                    final, note = det, f"model order rejected ({type(e).__name__}: {str(e)[:120]}) -> detect order"
            return pno, det, final, note
        with ThreadPoolExecutor(ORDER_WORKERS) as _ex:             # all pages at once
            res = list(_ex.map(_one, sorted(by_page)))
        out = []
        for pno, det, final, note in res:
            moved = sum(1 for x, y in zip(det, final) if x is not y)
            print(f"  page {pno}: {note}\n     detect: {short(det)}\n     final : {short(final)}   ({moved} position(s) changed)")
            out.extend(final)
        return out

    # ── called after the main chunk loop (formatted_by_id complete) ─────────
    def _validate_final_order(expected, formatted_by_id):
        exp_ids = [b["block_id"] for b in expected]
        got     = [i for i in formatted_by_id if i in set(exp_ids)]
        if got == exp_ids:
            print(f"✓ final order check: all {len(exp_ids)} sent blocks are in the planned order")
            return True
        miss = [i for i in exp_ids if i not in formatted_by_id]
        first = next((k for k, (x, y) in enumerate(zip(exp_ids, got)) if x != y), None)
        print(f"✗ final order check FAILED: missing={miss}, first mismatch at position {first}: planned {exp_ids[first] if first is not None else '-'} vs got {got[first] if first is not None else '-'}")
        return False


    # ── collect blocks flat (text + picture + empty-text) for chunk assembly ─
    def _collect_blocks_from_page_dicts(page_dicts):
        out = []
        for page in page_dicts:
            for region in page["regions"]:
                is_pic = region.get("type") == "picture"
                out.append({
                    "block_id":    region["block_id"],
                    "page":        region["page"],
                    "bodhan_type": region.get("bodhan_type", "Picture" if is_pic else "Text"),
                    "bbox_xyxy":   region.get("bbox_xyxy"),
                    "page_wh":     region.get("page_wh"),
                    "image_path":  region.get("image_path"),
                    "text":        "" if is_pic else (region.get("text", "") or ""),
                    "is_picture":  is_pic,
                })
        return out


    # ── serialize chunk → one <BLOCK id="..." ...> envelope per block ───────
    _HTML_TABLE_RE = re.compile(r"<\s*table\b", re.I)

    def _esc(text):
        return text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")

    def _bbox_norm_str(b):
        bb, wh = b.get("bbox_xyxy"), b.get("page_wh")
        if not bb or not wh: return "null"
        W, H = wh
        try:
            return f"{bb[0]/W:.3f},{bb[1]/H:.3f},{bb[2]/W:.3f},{bb[3]/H:.3f}"
        except Exception:
            return "null"

    def _serialize_blocks_for_combined(blocks):
        parts = []
        for b in blocks:
            body = "" if b.get("is_picture") else (b.get("text") or "").strip()
            if body and not _HTML_TABLE_RE.search(body):
                body = _esc(body)
            bbox_s = _bbox_norm_str(b)
            parts.append(
                f'<BLOCK id="{b["block_id"]}" type="{b["bodhan_type"]}" '
                f'page="{b["page"]}" bbox="{bbox_s}">\n{body}\n</BLOCK>'
            )
        return "\n\n".join(parts)


    # ── lenient deserializer: preserves EMIT ORDER (= reading order) ─────────
    _BLOCK_OPEN_RE  = re.compile(r"<BLOCK\b[^>]*?\bid\s*=\s*[\"']?(?P<id>[^\s\"'>]+)[\"']?[^>]*>", re.I)
    _BLOCK_CLOSE_RE = re.compile(r"</BLOCK\s*>", re.I)

    def _deserialize_combined_output(text):
        """Returns an insertion-ordered dict {block_id: markup}.
    Insertion order = the order the model emitted the wrappers = reading order."""
        text = re.sub(r"<think>.*?</think>", "", text or "", flags=re.S | re.I).strip()
        text = re.sub(r"^```[a-zA-Z]*\s*\n", "", text)
        text = re.sub(r"\n```\s*$", "", text)
        opens = list(_BLOCK_OPEN_RE.finditer(text))
        out = {}
        for i, m in enumerate(opens):
            end = opens[i + 1].start() if i + 1 < len(opens) else len(text)
            body = text[m.end():end]
            cm = _BLOCK_CLOSE_RE.search(body)
            if cm: body = body[:cm.start()]
            bid = m.group("id")
            if bid not in out:                 # keep first (= earliest) emit position
                out[bid] = body.strip()
        return out


    # ── tag grammar used by the fidelity skeleton (DOCX builder reuses it) ──
    _KNOWN_TAGS = "P|STYLE|BR|BLANK|LIST|ITEM|QUESTION|OPTIONS|OPTION|TABLE|ROW|CELL"
    _TAG_RE = re.compile(
        r"<(?P<close>/?)(?P<name>%s)(?P<attrs>(?:\s+[A-Za-z_][\w-]*\s*=\s*(?:\"[^\"]*\"|'[^']*'|[^\s<>/\"']+))*)\s*(?P<self>/?)>"
        % _KNOWN_TAGS, re.I)
    _INLINE_STYLE_RE = re.compile(r"</?STYLE(?:\s[^<>]*)?/?>", re.I)
    _ATTR_RE = re.compile(r"""([A-Za-z_][\w-]*)\s*=\s*("[^"]*"|'[^']*'|[^\s<>/"']+)""")
    _VOID_TAGS = {"BR", "BLANK"}

    def _parse_attrs(s):
        return {k.lower(): v.strip("\"'") for k, v in _ATTR_RE.findall(s or "")}


    # ── fidelity: multiset character comparison (reorder-insensitive) ───────
    def _skeleton(s):
        s = s.replace("\\n", "")                                  # literal backslash-n artefacts
        s = re.sub(r"\s+", "", s)                                 # whitespace edits allowed
        s = re.sub(r"([._\-–—…])\1{2,}", "", s)                   # blank runs
        for ch in FID_IGNORE_CHARS: s = s.replace(ch, "")
        return s

    def _skeleton_raw(raw):
        raw = raw or ""
        if _HTML_TABLE_RE.search(raw):
            s = re.sub(r"<br\s*/?>", " ", raw, flags=re.I)
            s = re.sub(r"</?[A-Za-z][^<>]*>", " ", s)
            s = _html.unescape(s)
        else:
            s = raw
        return _skeleton(s)

    _DROP_RE = re.compile(r"<DROP>(.*?)</DROP>", re.S | re.I)
    def _strip_drop(m):
        return _DROP_RE.sub("", m or "")

    def _skeleton_out(out):
        out = re.sub(r"</?DROP>", "", out or "", flags=re.I)   # dropped text still counts as present
        s = _INLINE_STYLE_RE.sub("", out or "")                   # m<STYLE>2</STYLE> → "m2"
        s = _TAG_RE.sub(" ", s)
        return _skeleton(_html.unescape(s))

    def _diff_multiset(a, b):
        """Reorder-insensitive diff: count chars dropped / added as multiset deltas."""
        if a == b: return 0, 0, []
        A, B = Counter(a), Counter(b)
        dropped = sum((A - B).values())
        added   = sum((B - A).values())
        sample  = []
        # show a handful of chars that disappeared / appeared, for debugging
        disappeared = [c for c in (A - B).elements()][:12]
        appeared    = [c for c in (B - A).elements()][:12]
        if disappeared: sample.append("dropped chars: " + "".join(disappeared))
        if appeared:    sample.append("added chars: "   + "".join(appeared))
        return dropped, added, sample

    def _chunk_faithful(blocks, parsed_ordered):
        """Chunk-level fidelity on multisets of characters.
    Only text blocks count (pictures have no OCR text)."""
        text_blocks = [b for b in blocks if not b.get("is_picture")]
        a = "".join(_skeleton_raw(b.get("text")) for b in text_blocks)
        b = "".join(_skeleton_out(parsed_ordered.get(x["block_id"], "")) for x in text_blocks)
        dropped, added, ex = _diff_multiset(a, b)
        # len(a) == 0 edge case: a chunk whose text input is empty has
        # 3% × 0 = 0, so even one added char would trip the check. The
        # min-char floor already handles that; keep this comment to make it
        # explicit for future readers.
        if dropped > max(FID_DROP_MIN_CHARS, FID_DROP_TOL * len(a)):
            return False, f"dropped {dropped}/{len(a)} chars: " + " | ".join(ex)
        if added > max(FID_ADD_MIN_CHARS, FID_ADD_TOL * len(a)):
            return False, f"added {added} chars (input {len(a)}): " + " | ".join(ex)
        return True, f"-{dropped}/+{added} chars vs {len(a)}"


    # ── raw-text fallback (if the chunk is unrecoverable) → tag markup ──────
    _HTML_TABLE_FULL_RE = re.compile(r"<table\b.*?</table>", re.S | re.I)

    def _html_table_to_markup(table_html):
        out = ['<TABLE border="1">']
        for row in re.findall(r"<tr\b[^>]*>(.*?)</tr>", table_html, flags=re.S | re.I):
            out.append("<ROW>")
            for tag, attrs, cell in re.findall(r"<(t[dh])\b([^>]*)>(.*?)</\1>", row, flags=re.S | re.I):
                cell = re.sub(r"<br\s*/?>", " ", cell, flags=re.I)
                cell = _html.unescape(re.sub(r"<[^>]+>", "", cell)).strip()
                cs = re.search(r"colspan\s*=\s*[\"']?(\d+)", attrs, re.I)
                extra = (' colspan="%s"' % cs.group(1)) if cs else ""
                if tag.lower() == "th": extra += ' bold="true"'
                out.append("<CELL%s>%s</CELL>" % (extra, _html.escape(cell, quote=False)))
            out.append("</ROW>")
        out.append("</TABLE>")
        return "\n".join(out)

    def _raw_to_markup(raw):
        raw = (raw or "").strip()
        if not _HTML_TABLE_RE.search(raw):
            return _html.escape(raw, quote=False)
        parts, pos = [], 0
        for m in _HTML_TABLE_FULL_RE.finditer(raw):
            if m.start() > pos: parts.append(_html.escape(raw[pos:m.start()], quote=False))
            parts.append(_html_table_to_markup(m.group(0)))
            pos = m.end()
        if pos < len(raw): parts.append(_html.escape(raw[pos:], quote=False))
        return "\n".join(parts)

    def _raw_fallback(b):
        return "" if b.get("is_picture") else _raw_to_markup(b.get("text") or "")


    # ── page image data URI (cached) ────────────────────────────────────────
    _PAGE_IMG_CACHE = {}

    def _page_image_data_uri(page_1based, max_side=800, jpeg_quality=85):
        if page_1based in _PAGE_IMG_CACHE:
            return _PAGE_IMG_CACHE[page_1based]
        img = render_page(page_1based - 1)
        if img is None:
            raise RuntimeError(f"render_page({page_1based - 1}) returned None")
        h, w = img.shape[:2]
        scale = min(1.0, max_side / max(h, w))
        if scale < 1.0:
            img = cv2.resize(img, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_AREA)
        ok, buf = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, jpeg_quality])
        if not ok:
            raise RuntimeError(f"cv2.imencode failed for page {page_1based}")
        uri = "data:image/jpeg;base64," + base64.b64encode(buf.tobytes()).decode("ascii")
        _PAGE_IMG_CACHE[page_1based] = uri
        return uri

    def _pages_in_chunk(blocks_chunk):
        return sorted({b["page"] for b in blocks_chunk})


    # ── HTTP with exponential backoff ───────────────────────────────────────
    class CombinedNetworkError(RuntimeError):
        pass

    def _post_with_retry(body, timeout):
        last = None
        for attempt in range(1, HTTP_ATTEMPTS + 1):
            try:
                r = requests.post(OPENROUTER_URL,
                                  headers={"Authorization": f"Bearer {OPENROUTER_KEY}",
                                           "Content-Type": "application/json"},
                                  json=body, timeout=timeout)
                if r.status_code == 429 or r.status_code >= 500:
                    last = f"HTTP {r.status_code}"
                else:
                    r.raise_for_status()
                    return r.json()
            except requests.exceptions.HTTPError:
                raise
            except requests.exceptions.RequestException as e:
                last = f"{type(e).__name__}: {str(e)[:100]}"
            if attempt < HTTP_ATTEMPTS:
                wait = min(30, 2 ** attempt)
                print(f"      ⚠ API call failed ({last}); retry {attempt}/{HTTP_ATTEMPTS - 1} in {wait}s")
                time.sleep(wait)
        raise CombinedNetworkError(f"gave up after {HTTP_ATTEMPTS} attempts: {last}")


    # ── one combined call → ordered {block_id: markup}, finish_reason ───────
    def _combined_once(blocks_chunk, model, temperature=0.0, timeout=240):
        tagged = _serialize_blocks_for_combined(blocks_chunk)
        pages = _pages_in_chunk(blocks_chunk)
        text_part = (
            COMBINED_PROMPT + tagged
            + f"\n\n[The {len(pages)} page image(s) attached below show page(s) {pages} "
              f"of the original scan. Use them for (A) verifying every removal and (C) formatting only — "
              f"the block order is fixed and must be kept. Do NOT read, transcribe or add any text from the "
              f"image(s). Each block's output may contain ONLY the words of that block.]"
        )
        content = [{"type": "text", "text": text_part}]
        for pno in pages:
            try:
                content.append({"type": "image_url", "image_url": {"url": _page_image_data_uri(pno)}})
            except Exception as e:
                print(f"      ⚠ page image render failed for page {pno}: {type(e).__name__}: {e} — text-only for it")

        body = {"model": model, "temperature": temperature,
                "messages": [{"role": "user", "content": content}]}
        data = _post_with_retry(body, timeout)
        if "error" in data: raise RuntimeError(f"openrouter: {data['error']}")
        raw    = data["choices"][0]["message"]["content"] or ""
        finish = data["choices"][0].get("finish_reason", "?")
        parsed = _deserialize_combined_output(raw)

        # single-block call where the model returned the body without a wrapper
        if len(blocks_chunk) == 1 and not parsed and raw.strip():
            body_txt = re.sub(r"^```[a-zA-Z]*\s*\n|\n```\s*$", "", raw.strip())
            parsed = {blocks_chunk[0]["block_id"]: body_txt.strip()}

        expected = {b["block_id"] for b in blocks_chunk}
        for extra in list(parsed):
            if extra not in expected: parsed.pop(extra, None)
        return parsed, finish


    # ── combined call with model fallback (tries COMBINED_MODELS in order) ──
    def _combined_with_model_fallback(blocks_chunk, temperature=0.0):
        last_err = None
        for model in COMBINED_MODELS:
            try:
                print(f"      trying {model} …")
                parsed, finish = _combined_once(blocks_chunk, model, temperature)
                return parsed, finish, model
            except CombinedNetworkError:
                raise
            except Exception as e:
                print(f"      {model} failed: {type(e).__name__}: {str(e)[:160]}")
                last_err = e
        raise RuntimeError(f"all combined models failed: {last_err}")


    # ── chunk-level fidelity + split-on-reject (same shape as the previous
    #    formatter, but using the multiset diff and the combined prompt) ────
    def _format_chunk(blocks_chunk, depth=0, temperature=0.0):
        ind = "  " * depth
        ids = [b["block_id"] for b in blocks_chunk]
        parsed, finish, used_model = _combined_with_model_fallback(blocks_chunk, temperature)

        missing = [i for i in ids if i not in parsed]
        if missing:
            ok, why = False, f"no <BLOCK> returned for {missing} (finish_reason={finish!r})"
        else:
            ok, why = _chunk_faithful(blocks_chunk, parsed)

        if ok:
            # keep the model's emit order; return the list of (id, markup) pairs
            emit = [bid for bid in parsed if bid in set(ids)]
            if emit != ids:
                print(f"    {ind}ℹ formatter changed block order {emit} — ignored, keeping {ids}")
            for bid in ids:
                for frag in _DROP_RE.findall(parsed[bid]):
                    print(f"    {ind}✂ {bid}: model removed furniture {frag.strip()[:50]!r}")
            ordered = [(bid, _strip_drop(parsed[bid]).strip()) for bid in ids]   # order fixed upstream; furniture removed
            return ordered, used_model, []

        print(f"    {ind}⚠ chunk {ids} rejected: {why}")
        if SHOW_REJECT_DEBUG:
            for b in blocks_chunk:
                print(f"    {ind}    {b['block_id']} raw: {(b.get('text') or '')[:70]!r}")
                print(f"    {ind}    {b['block_id']} out: {parsed.get(b['block_id'], '<missing>')[:140]!r}")

        if depth >= MAX_SPLIT_DEPTH:
            print(f"    {ind}✗ giving up on {ids} — using RAW text (detection order kept)")
            return [(b["block_id"], _raw_fallback(b)) for b in blocks_chunk], used_model, list(ids)

        if len(blocks_chunk) > 1:
            mid = len(blocks_chunk) // 2
            groups, temp = [blocks_chunk[:mid], blocks_chunk[mid:]], temperature
        else:
            groups, temp = [blocks_chunk], 0.3
        print(f"    {ind}↻ retrying as {len(groups)} group(s) (depth {depth + 1})")

        ordered_out, raw_ids = [], []
        model_used = used_model
        for g in groups:
            g_ordered, model_used, g_raw = _format_chunk(g, depth + 1, temp)
            ordered_out.extend(g_ordered)
            raw_ids.extend(g_raw)
        return ordered_out, model_used, raw_ids


    # ── build chunks from Stage 1 output ────────────────────────────────────
    blocks_A = _collect_blocks_from_page_dicts(page_dicts)
    print(f"total blocks: {len(blocks_A)} "
          f"({sum(1 for b in blocks_A if not b['is_picture'])} text, "
          f"{sum(1 for b in blocks_A if b['is_picture'])} picture)")

    # Skip empty-text, non-picture blocks — Bodhan sometimes returns a bbox with
    # text="" (typically a PageHeader box that wrapped no OCR content). Sending
    # these to the model is wasted work AND it will transcribe the image region
    # into the empty wrapper, which the fidelity check then (correctly) rejects.
    # These blocks pass straight through to Stage 3 as empty.
    skipped_empty = [b for b in blocks_A if (not b["is_picture"]) and not (b.get("text") or "").strip()]
    sendable      = [b for b in blocks_A if b["is_picture"] or (b.get("text") or "").strip()]
    sendable = _order_all_pages(sendable)

    if skipped_empty:
        print(f"skipping {len(skipped_empty)} empty-text non-picture block(s) "
              f"(bodhan returned a bbox but no OCR text — "
              f"will render as empty in the DOCX):")
        for b in skipped_empty:
            bb = b.get("bbox_xyxy")
            bb_s = "null" if not bb else f"[{int(bb[0])},{int(bb[1])},{int(bb[2])},{int(bb[3])}]"
            print(f"  {b['block_id']:<14} page {b['page']}  "
                  f"bodhan_type={b['bodhan_type']:<14} bbox={bb_s}")

    # chunk mode A — fixed CHUNK_BLOCKS per call. Pictures are included so the
    # model orders them alongside text blocks. Empty-text blocks are NOT sent.
    chunks = [sendable[i:i + CHUNK_BLOCKS] for i in range(0, len(sendable), CHUNK_BLOCKS)]
    print(f"combined chunks: {len(chunks)} ({CHUNK_BLOCKS} block(s) each, pictures included)")

    # ── preflight: can we even reach the API host? ──────────────────────────
    try:
        socket.gethostbyname(re.sub(r"^https?://([^/]+).*$", r"\1", OPENROUTER_URL))
    except OSError as e:
        msg = f"DNS lookup for the API host failed ({e}). Check your internet/VPN/DNS and re-run."
        if ABORT_IF_NO_DNS:
            raise RuntimeError(msg)
        print("⚠ " + msg)


    # ── per-chunk disk cache (keyed by prompt + model + chunk composition) ──
    # Keying includes every block_id and text in the chunk, so re-chunking
    # (changing CHUNK_BLOCKS) safely invalidates stale entries.
    _cache_path = METHOD_DIR / f"{stem}__combined_cache.json"
    def _chunk_key(chunk):
        sig = "|".join(f"{b['block_id']}:{(b.get('text') or '')[:400]}" for b in chunk)
        primary_model = COMBINED_MODELS[0]
        h = hashlib.sha1(f"{_PROMPT_HASH}|{primary_model}|{len(chunk)}|{sig}".encode("utf-8")).hexdigest()
        return h[:24]
    _cache = {}
    if USE_CACHE and _cache_path.exists():
        try: _cache = json.loads(_cache_path.read_text(encoding="utf-8"))
        except Exception: _cache = {}
        print(f"cache: {len(_cache)} chunk(s) in {_cache_path.name} (stale keys ignored)")


    # ── main loop (parallel chunks; results merged in chunk order = reading order) ──
    set_stage("A_combined", "qwen_chunks")
    formatted_by_id = {}        # insertion order = overall reading order
    raw_fallback_ids = []
    combined_failures = []
    model_used_log = {}
    import threading
    _plock = threading.Lock()
    _net = {"streak": 0, "tripped": False}

    def _do_chunk(item):
        ci, chunk = item
        ck = _chunk_key(chunk)
        if USE_CACHE and isinstance(_cache.get(ck), dict) and "ordered" in _cache[ck]:
            print(f"  chunk {ci}/{len(chunks)}: cached")
            return ci, chunk, _cache[ck]["ordered"], None, [], True
        if _net["tripped"]:
            return ci, chunk, None, "breaker", [], False
        try:
            print(f"  chunk {ci}/{len(chunks)} ({len(chunk)} blocks, pages={_pages_in_chunk(chunk)}) ...")
            ordered, used_model, raw_ids = _format_chunk(chunk)
            with _plock:
                _net["streak"] = 0; model_used_log[ci] = used_model
            return ci, chunk, ordered, None, raw_ids, False
        except Exception as e:
            print(f"    chunk {ci} failed: {type(e).__name__}: {str(e)[:200]} -> keeping RAW text for its blocks")
            with _plock:
                if isinstance(e, CombinedNetworkError):
                    _net["streak"] += 1
                    if _net["streak"] >= NET_FAIL_BREAKER: _net["tripped"] = True
                else:
                    _net["streak"] = 0
            return ci, chunk, None, e, [], False

    with ThreadPoolExecutor(CHUNK_WORKERS) as _ex:
        _results = list(_ex.map(_do_chunk, enumerate(chunks, 1)))
    for ci, chunk, ordered, err, raw_ids, was_cached in _results:
        if ordered is not None:
            for bid, mk in ordered:
                if bid not in formatted_by_id: formatted_by_id[bid] = mk
            raw_fallback_ids.extend(raw_ids)
            if USE_CACHE and not was_cached: _cache[_chunk_key(chunk)] = {"ordered": ordered}
        else:
            for b in chunk:
                formatted_by_id[b["block_id"]] = _raw_fallback(b)
                raw_fallback_ids.append(b["block_id"])
            combined_failures.append(ci)
    breaker_tripped = _net["tripped"]
    if breaker_tripped:
        print(f"\n  STOP: {NET_FAIL_BREAKER}+ consecutive network failures - remaining chunks used RAW text. Fix the connection and re-run; cached chunks are reused.")
    if USE_CACHE:
        _cache_path.write_text(json.dumps(_cache, ensure_ascii=False), encoding="utf-8")

    # ── order validation: the formatter output must follow the planned order ──
    _validate_final_order(sendable, formatted_by_id)

    # Any block the model never emitted (shouldn't happen, but defensively):
    # Blocks the model never saw: empty-text non-picture blocks skipped above,
    # plus any residual the model forgot. Give them an empty markup — the
    # downstream builder will record them as "empty_text_or_block" and skip them.
    # Importantly these are NOT counted as raw-fallback (nothing was sent).
    for b in blocks_A:
        if b["block_id"] not in formatted_by_id:
            formatted_by_id[b["block_id"]] = ""   # renders as empty block

    # Attach formatted_text + build reading-order-sorted blocks list
    _block_by_id = {b["block_id"]: b for b in blocks_A}
    for bid, mk in formatted_by_id.items():
        if bid in _block_by_id:
            _block_by_id[bid]["formatted_text"] = mk

    ordered_blocks = [_block_by_id[bid] for bid in formatted_by_id if bid in _block_by_id]
    # any residual block missing from the model's output goes at the end in detection order
    for b in blocks_A:
        if b["block_id"] not in formatted_by_id:
            ordered_blocks.append(b)

    set_stage("-", "-")

    print(f"\n── Stage 2 summary ──")
    print(f"chunks: {len(chunks)}  ·  blocks sent: {sum(len(c) for c in chunks)}  "
          f"·  formatted: {len(formatted_by_id)}  ·  raw-fallback: {len(set(raw_fallback_ids))}")
    if combined_failures:
        print(f"  failed chunks (used RAW): {combined_failures}")

## Final — DOCX + sidecar

In [ ]:
# ─── FINAL: tags → DOCX + sidecar (all routes) ─────────────────────────────
set_stage("final", "build_docx")
docx_path = METHOD_DIR / f"{stem}__{ROUTE}.docx"
try:
    final_docx, block_meta = build_docx_with_blocks(ordered_blocks, docx_path)
except PermissionError:
    docx_path = docx_path.with_name(f"{docx_path.stem}_{datetime.datetime.now():%Y%m%d_%H%M%S}.docx")
    final_docx, block_meta = build_docx_with_blocks(ordered_blocks, docx_path)

sidecar = {
    "pdf_path": str(PDF_PATH), "docx_path": str(final_docx), "route": ROUTE, "route_info": ROUTE_INFO,
    "models": M, "pages_processed": [p + 1 for p in page_indices],
    "reading_order": [b["block_id"] for b in ordered_blocks],
    "blocks": {bid: {**m, "bbox_norm": next((b.get("bbox_norm") for b in ordered_blocks if b["block_id"] == bid), None)} for bid, m in block_meta.items()},
}
sidecar_path = Path(final_docx).with_suffix("").with_suffix(".blocks.json")
sidecar_path.write_text(json.dumps(sidecar, ensure_ascii=False, indent=2), encoding="utf-8")
set_stage("-", "-")

calls = [c for c in CALLS if c["engine"] != "_wallclock_"]
print(f"saved {final_docx}\nsaved {sidecar_path}")
print(f"route={ROUTE} · blocks={len(block_meta)} · rendered={sum(m['rendered'] for m in block_meta.values())}")
print(f"api calls={len(calls)} · tokens in/out={sum(c['tok_in'] for c in calls):,}/{sum(c['tok_out'] for c in calls):,} "
      f"· ${sum((c['cost'] or 0) for c in calls):.4f} · {sum(c['seconds'] for c in calls):.0f}s (summed over calls)")


saved compare_out\infinite_ocr\tmax__digital_english.docx
saved compare_out\infinite_ocr\tmax__digital_english.blocks.json
route=digital_english · blocks=279 · rendered=258
api calls=40 · tokens in/out=80,138/22,509 · $0.0719 · 236s (summed over calls)
